# SynerPlant v4.3 - all studies in the frozen extraction manifest

This copy removes the 380-study selection limit. The parsing, export, QA, checkpoint and resume functions are unchanged.

**Ready configuration:** `RUN_MODE = "MANIFEST"`, `EXECUTION_MODE = "BATCH"`, `RUN_ALL_STUDIES = True`, `BATCH_NO = 1`, `FORCE_REPROCESS = False`.

After the manifest is loaded and its existing filters are applied, the notebook automatically sets `BATCH_SIZE` to the exact number of distinct studies. Do not enter an approximate total such as 700. All selected studies and their manifest-listed sources are processed in manifest order. Studies absent from the manifest are not added automatically; class X remains excluded. A missing local source is reported, not silently dropped.

The storage name `docling_local_sourceqa_v4_3_rerun_380` and revision `v4_3_project_20260923_rerun_380` are intentionally retained for compatibility with the earlier fresh 380-study run. The number in these names is NOT a selection limit. Already completed, current sources in that fresh run are resumed; remaining sources are added. Old v4.2 results are neither deleted nor reused as the new run's outputs.

Stop the old notebook/kernel before starting this copy. Run all cells from top to bottom. After interruption, keep these settings and both run names unchanged; restart the same notebook from the top. Keep the frozen manifest unchanged during the run. Do not enable FORCE_REPROCESS to resume. A source interrupted before its export was saved may need to restart; completed sources with valid checkpoints and artifacts are skipped.

New results: `04_extracted_json/docling_local_sourceqa_v4_3_rerun_380/`.
Logs/checkpoints: `06_pipeline_logs/docling_local_sourceqa_v4_3_rerun_380/`.
Use each study's `STUDY_HANDOFF.v4_3.json` for corrected asset paths. The master Google spreadsheet and original corpus are not modified.

Selection and syntax tests do not establish PDF/OCR accuracy or Windows/Drive runtime compatibility. A completed batch does not mean every source passed QA. Inspect the source-action report and verify extracted values against the originals before database use.


In [ ]:
# 1. Dependencies: validate installed versions; never silently upgrade a working environment.
import importlib.util
import importlib.metadata
import subprocess
import sys

REQUIRED = {
    "docling": "docling>=2.129,<3", "fitz": "PyMuPDF>=1.26",
    "openpyxl": "openpyxl>=3.1.5,<4", "xlrd": "xlrd>=2.0.1,<3",
    "cv2": "opencv-python>=4.10", "pandas": "pandas>=2.2",
    "numpy": "numpy>=1.26", "PIL": "Pillow>=10", "tabulate": "tabulate>=0.9",
    "jsonschema": "jsonschema>=4.23", "packaging": "packaging>=23",
    "charset_normalizer": "charset-normalizer>=3",
}
AUTO_INSTALL_MISSING = True
missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError("Missing dependencies: " + ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()
from packaging.requirements import Requirement

def check_dependency_versions(required):
    bad, versions = [], {}
    aliases = {"opencv-python": ("opencv-python", "opencv-python-headless", "opencv-contrib-python", "opencv-contrib-python-headless")}
    for spec in required.values():
        req = Requirement(spec)
        installed = None
        for distribution in aliases.get(req.name, (req.name,)):
            try:
                installed = importlib.metadata.version(distribution)
                break
            except importlib.metadata.PackageNotFoundError:
                continue
        versions[req.name] = installed
        if installed is None or not req.specifier.contains(installed, prereleases=True):
            bad.append(f"{req.name}: installed={installed}; required={req.specifier}")
    return versions, bad

DEPENDENCY_VERSIONS, incompatible = check_dependency_versions(REQUIRED)
if incompatible:
    print("\n".join(incompatible))
    raise RuntimeError("Dependency versions do not match. Update the listed packages in this Python environment, then restart the kernel. No automatic upgrades were performed.")
print("Python:", sys.version)
print("Dependency versions:", DEPENDENCY_VERSIONS)


In [ ]:
# ============================================================
# 2. Imports + LOCAL WINDOWS paths
# ============================================================

import csv
import gc
import gzip
import hashlib
import json
import math
import mimetypes
import os
import platform
import re
import shutil
import subprocess
import time
import traceback
import zipfile
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import cv2
import fitz
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from openpyxl.utils import get_column_letter

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc import ImageRefMode

PROJECT_ROOT = Path("G:/\u041c\u043e\u0439 \u0434\u0438\u0441\u043a/TFM_CORPUS_NORMALIZED")

def windows_extended_path_text(value):
    import ntpath
    value = ntpath.abspath(str(value))
    slash = chr(92)
    prefix = slash * 2 + "?" + slash
    if value.startswith(prefix):
        return value
    if value.startswith(slash * 2):
        return prefix + "UNC" + slash + value[2:]
    return prefix + value

PROJECT_ROOT_DISPLAY = str(PROJECT_ROOT)
if os.name == "nt":
    PROJECT_ROOT = Path(windows_extended_path_text(PROJECT_ROOT))
FULLTEXT_DIR = PROJECT_ROOT / "01_fulltext"
SUPPLEMENTS_DIR = PROJECT_ROOT / "02_supplements"
MANIFEST_DIR = PROJECT_ROOT / "00_frozen_manifests"
EXTRACTED_DIR = PROJECT_ROOT / "04_extracted_json"
EXPORTS_DIR = PROJECT_ROOT / "05_exports"
LOGS_DIR = PROJECT_ROOT / "06_pipeline_logs"

print("Computer:", platform.node())
print("System:", platform.system())
print("Python:", sys.version.split()[0])
print("Project:", PROJECT_ROOT)

for name, p in {
    "PROJECT_ROOT": PROJECT_ROOT,
    "01_fulltext": FULLTEXT_DIR,
    "02_supplements": SUPPLEMENTS_DIR,
    "00_frozen_manifests": MANIFEST_DIR,
    "04_extracted_json": EXTRACTED_DIR,
    "06_pipeline_logs": LOGS_DIR,
}.items():
    print(f"{name:22s} -> {p.exists()}   {p}")

missing_dirs = [str(p) for p in [PROJECT_ROOT, FULLTEXT_DIR, SUPPLEMENTS_DIR] if not p.exists()]
if missing_dirs:
    raise RuntimeError("Required corpus folders are not visible:\n" + "\n".join(missing_dirs))

for p in [MANIFEST_DIR, EXTRACTED_DIR, EXPORTS_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("\nLOCAL WINDOWS MODE READY")


In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

PIPELINE_VERSION = "docling_local_sourceqa_v4_3_rerun_380"  # Independent run; keep this name when resuming.
EXTRACTOR_REVISION = "v4_3_project_20260923_rerun_380"
# UPGRADE_EXISTING: all already-created study folders in the current manifest;
# never invokes Docling.convert(). PDFs/DOCX are repaired from saved JSON.
# BATCH: process BATCH_NO; upgrades old results first, converts only when needed.
# AUDIT_EXISTING: read-only planning, no parsing or output modification.
EXECUTION_MODE = "BATCH"
QUALITY_DIR_NAME = "quality_v4_3"
REPARSE_SOURCE_IDS = []  # BATCH only: active file IDs for an explicit selective retry.
DELIMITED_ENCODING_OVERRIDES = {}  # e.g. {"supplement.csv": "cp1252"}
DELIMITED_SEPARATOR_OVERRIDES = {}  # e.g. {"supplement.csv": ";"}
# Full cell provenance is streamed; large sheets are not silently truncated.
STREAM_ALL_CELL_PROVENANCE = True
DATABASE_SPEC = "TFM_ML_DATABASE_V3_v3.4.0_corpus_v1.5"

# ------------------------------------------------------------
# RUN MODE
# ------------------------------------------------------------

# PILOT    = selected study IDs
# MANIFEST = production batches from frozen V3 manifest
RUN_MODE = "MANIFEST"

# Used only in PILOT mode
STUDY_IDS: List[str] = []

# ------------------------------------------------------------
# BATCH CONTROL
# ------------------------------------------------------------

# True: select ALL distinct studies after the frozen manifest is loaded.
# The exact BATCH_SIZE is calculated in section 5; no approximate total is needed.
RUN_ALL_STUDIES = True

# Only used as a manual limit when RUN_ALL_STUDIES = False.
BATCH_SIZE = 380

# Keep 1 for the all-manifest run, including when resuming after interruption.
BATCH_NO = 1

BATCH_START = (BATCH_NO - 1) * BATCH_SIZE

# ------------------------------------------------------------
# MANIFEST
# ------------------------------------------------------------

FROZEN_MANIFEST_PATH = (
    MANIFEST_DIR / "synerplant_current_extraction_manifest.csv"
)

REQUIRE_FILE_LEVEL_MANIFEST = True
EXCLUDE_CLASS_X = True

# IMPORTANT:
# Keep False for normal production.
# True means reprocess everything selected even if checkpoints exist.
FORCE_REPROCESS = False

HASH_POLICY = "ONCE"  # NONE / ONCE / ALWAYS

# ------------------------------------------------------------
# PARSER QUALITY
# ------------------------------------------------------------

IMAGE_SCALE = 1.5
FIGURE_RENDER_SCALE = 2.0

DO_OCR = True
TABLE_MODE_ACCURATE = True

TRY_VECTOR_BAR_DIGITIZATION = True
MIN_BAR_DIGITIZATION_R2 = 0.995

# Keep OFF for the first full corpus pass.
ENABLE_DOCLING_CHART_EXTRACTION = False

# ------------------------------------------------------------
# SPREADSHEETS
# ------------------------------------------------------------

EXPORT_CELL_LEVEL_PROVENANCE = True
MAX_NONEMPTY_CELLS_FOR_LONG_EXPORT = 500_000

# ------------------------------------------------------------
# MASTER DATABASE SAFETY
# ------------------------------------------------------------

WRITE_BACK_TO_MASTER = False
assert WRITE_BACK_TO_MASTER is False

# ------------------------------------------------------------
# OUTPUT DIRECTORIES
# ------------------------------------------------------------

RUN_EXTRACTED_DIR = EXTRACTED_DIR / PIPELINE_VERSION
RUN_LOGS_DIR = LOGS_DIR / PIPELINE_VERSION
CHECKPOINT_DIR = RUN_LOGS_DIR / "checkpoints"
BATCH_REPORT_DIR = RUN_LOGS_DIR / "batch_reports"
HASH_CACHE_PATH = RUN_LOGS_DIR / "file_hash_cache.csv"

for p in [
    RUN_EXTRACTED_DIR,
    RUN_LOGS_DIR,
    CHECKPOINT_DIR,
    BATCH_REPORT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("Pipeline:", PIPELINE_VERSION)
print("Database spec:", DATABASE_SPEC)
print("Run mode:", RUN_MODE)
print("Batch number:", BATCH_NO)
if RUN_ALL_STUDIES:
    print("Selection: ALL studies in the filtered frozen manifest; exact count follows in section 5.")
else:
    print("Batch size:", BATCH_SIZE)
    print("Batch range:", BATCH_START + 1, "to", BATCH_START + BATCH_SIZE)
print("Manifest:", FROZEN_MANIFEST_PATH)
print("Manifest exists:", FROZEN_MANIFEST_PATH.exists())
print("Output:", RUN_EXTRACTED_DIR)
print("Force reprocess:", FORCE_REPROCESS)

In [ ]:
# ============================================================
# 4. Project-aligned scientific extraction specification (V3 + BIOTIC extension)
# ============================================================

SCREENING_CLASSES = {
    "A": {"role": "PRIMARY", "definition": "tomato + stress + biostimulant"},
    "B": {"role": "BASELINE_STRESS", "definition": "tomato + stress"},
    "C": {"role": "BIOSTIMULANT_ONLY", "definition": "tomato + biostimulant"},
    "D": {"role": "TRANSFER", "definition": "other crop + stress + biostimulant"},
    "E": {"role": "AUXILIARY", "definition": "other crop with only stress or only biostimulant"},
    "X": {"role": "EXCLUDE", "definition": "outside modeling task"},
}

PRIMARY_TARGET = "strict_SxB = (SB - S) - (B - C) in matched tomato C/S/B/SB experiments"
SECONDARY_TARGET = "SB_vs_S direct mitigation"
PROJECT_SUCCESS_CRITERION = "retain the evidence needed to complete the SynerPlant/TFM objectives, not merely parse PDFs"

SOURCE_PRIORITY = [
    "SUPPLEMENT_MACHINE_READABLE", "SUPPLEMENT_TABLE", "MAIN_TEXT_TABLE",
    "EXPLICIT_PROSE_VALUE", "FIGURE_DIGITIZED",
]

ML_ROLES = [
    "ID_GROUP", "PROVENANCE_ONLY", "CONTEXT_ONLY", "FEATURE",
    "QUALITY_FEATURE", "TARGET", "POST_TREATMENT_MECHANISTIC",
]

OUTCOME_FAMILIES = [
    "dry_biomass", "fresh_biomass", "yield", "growth_morphology", "root_architecture",
    "photosynthesis", "gas_exchange", "chlorophyll_fluorescence", "pigments", "water_relations",
    "ions_nutrients", "oxidative_damage", "antioxidants_redox", "osmolytes", "hormones_signalling",
    "enzyme_activities", "metabolomics", "transcriptomics", "proteomics", "fruit_quality", "phenology",
    "survival_mortality", "disease_incidence", "disease_severity", "disease_progress", "lesion_symptoms",
    "pathogen_biomass_load", "viral_load_titer", "nematode_infection_reproduction", "pest_damage",
    "infection_traits", "stress_indices", "other",
]

TIMEPOINT_ANCHORS = [
    "AFTER_STRESS_ONSET", "AFTER_INOCULATION", "AFTER_BIOSTIMULANT",
    "AFTER_TRANSPLANT", "PLANT_AGE", "HARVEST", "OTHER",
]

EXTRACTION_TARGETS = {
    "publication_study": ["doi", "title", "authors", "year", "first_publication_date", "journal", "crop", "species", "cultivar", "genotype", "country", "dataset_role", "temporal_eligibility"],
    "experiment_context": [
        "growth_stage", "plant_age", "host_resistance_status", "site", "season", "environment_type", "growth_facility",
        "soil_or_soilless", "soil_texture", "substrate_composition", "pot_volume_l", "plant_density", "spacing",
        "photoperiod_h", "ppfd_umol_m2_s", "light_source", "light_spectrum", "day_temperature_c",
        "night_temperature_c", "temperature_regime_raw", "relative_humidity_pct", "vpd_kpa", "co2_ppm",
        "soil_ph", "substrate_ph", "soil_ec_ds_m", "substrate_ec_ds_m", "organic_matter_pct",
        "irrigation_method", "irrigation_regime", "nutrient_solution", "fertilization", "experimental_design",
        "randomization", "blocking", "experimental_unit", "biological_replicates", "technical_replicates", "sampling_design"
    ],
    "stress_factors": [
        "stress_domain", "stress_type_raw", "stress_type_standard", "stress_level_raw", "agent_raw", "duration", "onset", "cycles", "recovery",
        "NaCl", "EC", "irrigation_fraction", "field_capacity", "soil_water_content", "water_potential", "temperature", "control_temperature",
        "nutrient_concentration", "toxic_element", "light_UV", "waterlogging_hypoxia",
        "disease_name_raw", "disease_name_standard", "pathogen_group", "genus", "species", "subspecies", "forma_specialis", "pathovar", "race_biotype", "strain", "isolate",
        "inoculum_material", "inoculum_concentration", "inoculum_unit", "inoculum_volume_amount", "inoculation_method", "inoculation_site",
        "inoculation_timing", "timing_relative_to_biostimulant", "post_inoculation_conditions", "infection_confirmation_method"
    ],
    "biostimulant": [
        "component_name", "class", "source", "source_organism", "strain", "formulation", "carrier", "composition",
        "active_compounds", "extraction_production_method", "manufacturer", "commercial_product", "dose", "concentration",
        "application_route", "application_site", "application_stage", "timing", "timing_relative_to_stress", "frequency", "application_count"
    ],
    "measurements": [
        "outcome", "family", "value_type", "organ", "tissue", "timepoint", "timepoint_anchor", "method", "instrument", "assay",
        "assessment_definition", "scale_min", "scale_max", "denominator", "mean", "median", "SD", "SE", "CI", "n", "n_definition",
        "unit", "p_value", "significance", "transformation", "provenance"
    ],
    "analysis_support": ["reported_effects", "factorial_ANOVA_or_model", "interaction_p_value", "shared_controls", "dependency_clusters", "risk_of_bias"],
}

spec_path = RUN_LOGS_DIR / "EXTRACTION_SPEC_V4_3.json"
if EXECUTION_MODE != "AUDIT_EXISTING":
    spec_path.write_text(json.dumps({
        "pipeline_version": PIPELINE_VERSION, "database_spec": DATABASE_SPEC,
        "project_success_criterion": PROJECT_SUCCESS_CRITERION, "screening_classes": SCREENING_CLASSES,
        "primary_target": PRIMARY_TARGET, "secondary_target": SECONDARY_TARGET,
        "stress_scope": "ABIOTIC + BIOTIC; multiple stress factors per arm are allowed",
        "source_priority": SOURCE_PRIORITY, "ml_roles": ML_ROLES, "outcome_families": OUTCOME_FAMILIES,
        "timepoint_anchors": TIMEPOINT_ANCHORS, "extraction_targets": EXTRACTION_TARGETS,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
print("Extraction spec:", spec_path)


In [ ]:
# Manifest diagnostics (read the same manifest that the extractor will use).
print("Manifest:", FROZEN_MANIFEST_PATH)
print("Exists:", FROZEN_MANIFEST_PATH.exists())


In [ ]:
# ============================================================
# 5. Local corpus index + frozen manifest handling
# ============================================================

def clean_str(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x).strip()


def study_id_from_filename(name: str) -> Optional[str]:
    m = re.match(r"^(\d+)__", name)
    if m:
        return str(int(m.group(1)))
    m = re.match(r"^(D_[A-Za-z0-9]+)__", name)
    if m:
        return m.group(1)
    m = re.match(r"^(U_[A-Za-z0-9]+)__", name)
    if m:
        return m.group(1)
    return None


def role_from_folder(path: Path) -> str:
    if FULLTEXT_DIR in path.parents or path.parent == FULLTEXT_DIR:
        return "FULLTEXT"
    if SUPPLEMENTS_DIR in path.parents or path.parent == SUPPLEMENTS_DIR:
        return "SUPPLEMENT"
    return "OTHER"


def guess_mime(path: Path) -> str:
    return mimetypes.guess_type(path.name)[0] or "application/octet-stream"


def build_local_corpus_index() -> pd.DataFrame:
    rows = []
    for base in [FULLTEXT_DIR, SUPPLEMENTS_DIR]:
        for p in sorted(base.rglob("*")):
            if not p.is_file():
                continue
            sid = study_id_from_filename(p.name)
            if not sid:
                continue
            st = p.stat()
            rows.append({
                "study_id": sid,
                "role": role_from_folder(p),
                "source_path": str(p),
                "source_file_name": p.name,
                "mime_type": guess_mime(p),
                "size_bytes": int(st.st_size),
                "modified_time_ns": int(st.st_mtime_ns),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No normalized corpus files were found.")
    rank = {"FULLTEXT": 0, "SUPPLEMENT": 1}
    df["_role_rank"] = df["role"].map(rank).fillna(9)
    df = df.sort_values(["study_id", "_role_rank", "source_file_name"], kind="stable")
    return df.drop(columns="_role_rank")


def load_frozen_manifest(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Production MANIFEST mode requires a frozen manifest: {path}\n"
            "Create it from TFM_ML_DATABASE_V3; do not substitute folder ordering."
        )
    if path.suffix.lower() == ".csv":
        m = pd.read_csv(path, dtype=str, keep_default_na=False)
    elif path.suffix.lower() == ".json":
        obj = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(obj, list):
            records = obj
        elif isinstance(obj, dict):
            records = obj.get("files", obj.get("records", []))
        else:
            raise ValueError("JSON manifest must be a list or an object with files/records")
        if not isinstance(records, list) or any(not isinstance(r, dict) for r in records):
            raise ValueError("Manifest records must be a list of objects")
        m = pd.DataFrame(records).fillna("")
    else:
        raise ValueError("Manifest must be CSV or JSON")
    if "study_id" not in m.columns:
        raise ValueError("Manifest requires study_id")
    m["study_id"] = m["study_id"].astype(str).str.strip()
    if m["study_id"].eq("").any():
        raise ValueError("Manifest contains empty study_id values")
    if "source_file_name" in m.columns:
        if m["source_file_name"].astype(str).str.strip().eq("").any():
            raise ValueError("File-level manifest contains an empty filename")
        key_cols = [c for c in ["study_id", "role", "source_file_name"] if c in m]
        if m.duplicated(key_cols).any():
            raise ValueError("Duplicate source records in frozen manifest; resolve explicitly before extraction")
    if "source_file_id" in m and m["source_file_id"].astype(str).str.strip().ne("").any():
        ids = m.loc[m["source_file_id"].astype(str).str.strip().ne(""), "source_file_id"]
        if ids.duplicated().any():
            raise ValueError("A source_file_id appears more than once in the manifest; review study/file identity")
    if EXCLUDE_CLASS_X and "screening_class" in m.columns:
        m = m[~m["screening_class"].str.upper().eq("X")].copy()
    if "batch_order" in m.columns:
        m["_batch_order_num"] = pd.to_numeric(m["batch_order"], errors="coerce")
        m = m.sort_values(["_batch_order_num"], kind="stable").drop(columns="_batch_order_num")
    return m.reset_index(drop=True)


corpus_df = build_local_corpus_index()
print("Indexed local files:", len(corpus_df))
print("Fulltexts:", int(corpus_df["role"].eq("FULLTEXT").sum()))
print("Supplements:", int(corpus_df["role"].eq("SUPPLEMENT").sum()))
print("Studies with fulltext:", corpus_df.loc[corpus_df["role"].eq("FULLTEXT"), "study_id"].nunique())

manifest_df = None
if RUN_MODE.upper() == "MANIFEST":
    manifest_df = load_frozen_manifest(FROZEN_MANIFEST_PATH)
    if REQUIRE_FILE_LEVEL_MANIFEST and "source_file_name" not in manifest_df.columns:
        raise ValueError("Production manifest must be file-level and include source_file_name")
    print("Manifest records:", len(manifest_df))
    display(manifest_df.head(20))
elif RUN_MODE.upper() == "PILOT":
    if not STUDY_IDS:
        raise ValueError("PILOT mode requires explicit STUDY_IDS")
else:
    raise ValueError("RUN_MODE must be PILOT or MANIFEST")

# Resolve the all-manifest selection only after loading/filtering the manifest.
# This changes selection only; extraction settings and checkpoint keys are unchanged.
if RUN_ALL_STUDIES:
    if RUN_MODE.upper() != "MANIFEST":
        raise ValueError("RUN_ALL_STUDIES requires RUN_MODE = MANIFEST")
    if BATCH_NO != 1:
        raise ValueError("Keep BATCH_NO = 1 when RUN_ALL_STUDIES = True")
    BATCH_SIZE = int(manifest_df["study_id"].nunique())
    BATCH_START = 0
    if BATCH_SIZE == 0:
        raise ValueError("No studies remain in the filtered frozen manifest")
    print("ALL MANIFEST STUDIES:", BATCH_SIZE)
    print("Manifest source files (fulltexts and supplements):", len(manifest_df))


In [ ]:
# ============================================================
# 6. Deterministic study/source selection
# ============================================================

def selected_study_ids() -> List[str]:

    # --------------------------------------------------------
    # PILOT
    # --------------------------------------------------------
    if RUN_MODE.upper() == "PILOT":

        ids = [
            clean_str(x)
            for x in STUDY_IDS
            if clean_str(x)
        ]

        missing = [
            sid
            for sid in ids
            if not (
                (corpus_df["study_id"].eq(sid))
                & corpus_df["role"].eq("FULLTEXT")
            ).any()
        ]

        if missing:
            raise RuntimeError(
                f"FULLTEXT not found for pilot study IDs: {missing}"
            )

        return ids[:BATCH_SIZE]

    # --------------------------------------------------------
    # MANIFEST
    # --------------------------------------------------------

    # Preserve first appearance in frozen manifest.
    # Never sort production studies by filename or study ID.
    seen = set()
    ordered = []

    for sid in manifest_df["study_id"].tolist():

        sid = clean_str(sid)

        if sid and sid not in seen:
            seen.add(sid)
            ordered.append(sid)

    start = BATCH_START
    end = BATCH_START + BATCH_SIZE

    return ordered[start:end]


def source_rows_for_study(study_id: str) -> pd.DataFrame:

    local = corpus_df[
        corpus_df["study_id"].eq(study_id)
    ].copy()

    # --------------------------------------------------------
    # PILOT
    # --------------------------------------------------------

    if RUN_MODE.upper() == "PILOT":

        rank = {
            "FULLTEXT": 0,
            "SUPPLEMENT": 1,
        }

        local["_rank"] = (
            local["role"]
            .map(rank)
            .fillna(9)
        )

        return (
            local
            .sort_values(
                ["_rank", "source_file_name"],
                kind="stable",
            )
            .drop(columns="_rank")
        )

    # --------------------------------------------------------
    # MANIFEST
    # --------------------------------------------------------

    m = manifest_df[
        manifest_df["study_id"].eq(study_id)
    ].copy()

    if m.empty:
        return pd.DataFrame()

    if (
        "source_file_name" not in m.columns
        or not m["source_file_name"]
        .astype(str)
        .str.strip()
        .any()
    ):

        if REQUIRE_FILE_LEVEL_MANIFEST:
            raise RuntimeError(
                f"Study {study_id}: "
                "file-level manifest required"
            )

        return local

    rows = []

    for order, rec in enumerate(
        m.to_dict("records"),
        1,
    ):

        fn = clean_str(
            rec.get("source_file_name")
        )

        matches = local[local["source_file_name"].eq(fn)]
        manifest_role = clean_str(rec.get("role")).upper()
        if manifest_role:
            matches = matches[matches["role"].eq(manifest_role)]
        if len(matches) > 1:
            raise RuntimeError(f"Ambiguous local filename for {study_id}: {fn}")

        if matches.empty:

            rows.append({
                "study_id": study_id,
                "role": (
                    clean_str(rec.get("role"))
                    or "UNKNOWN"
                ),
                "source_path": "",
                "source_file_name": fn,
                "mime_type": clean_str(
                    rec.get("mime_type")
                ),
                "size_bytes": 0,
                "modified_time_ns": 0,
                "manifest_missing_local": True,
                "source_file_id": clean_str(
                    rec.get("source_file_id")
                ),
                "screening_class": clean_str(
                    rec.get("screening_class")
                ),
                "ml_role": clean_str(
                    rec.get("ml_role")
                ),
                "source_rank": clean_str(
                    rec.get("source_rank")
                ),
                "manifest_order": order,
            })

        else:

            row = matches.iloc[0].to_dict()

            row.update({
                "manifest_missing_local": False,
                "source_file_id": clean_str(
                    rec.get("source_file_id")
                ),
                "screening_class": clean_str(
                    rec.get("screening_class")
                ),
                "ml_role": clean_str(
                    rec.get("ml_role")
                ),
                "source_rank": clean_str(
                    rec.get("source_rank")
                ),
                "manifest_order": order,
            })

            # Current manifest source_file_id is the active corpus ID used by extraction_queue.
            # If a future manifest carries active_file_id explicitly, preserve both IDs.
            local_values = {k: row[k] for k in ["source_path", "size_bytes", "modified_time_ns", "mime_type", "role"] if k in row}
            row.update(rec)
            row.update(local_values)
            if clean_str(rec.get("active_file_id")):
                row["historical_source_file_id"] = clean_str(rec.get("source_file_id"))
                row["source_file_id"] = clean_str(rec["active_file_id"])
            row["manifest_missing_local"] = False
            row["manifest_order"] = order
            rows.append(row)

    for i, row in enumerate(rows):
        for key, value in m.iloc[i].to_dict().items():
            row.setdefault(key, value)
    return pd.DataFrame(rows)


SELECTED_STUDIES = selected_study_ids()
if EXECUTION_MODE in {"UPGRADE_EXISTING", "AUDIT_EXISTING"}:
    pool = (manifest_df["study_id"].drop_duplicates().tolist()
            if RUN_MODE.upper() == "MANIFEST" else list(STUDY_IDS))
    SELECTED_STUDIES = [sid for sid in pool if (RUN_EXTRACTED_DIR / str(sid)).is_dir()]
if EXECUTION_MODE not in {"BATCH", "UPGRADE_EXISTING", "AUDIT_EXISTING"}:
    raise ValueError("Unknown EXECUTION_MODE")
print("Execution mode:", EXECUTION_MODE)
if EXECUTION_MODE != "BATCH":
    print("Existing folders selected; BATCH_NO does not limit this maintenance pass.")

print()
print("=" * 70)
print(
    f"BATCH {BATCH_NO} | "
    f"{len(SELECTED_STUDIES)} studies"
)
print("=" * 70)

print("Selected studies:")
print(SELECTED_STUDIES)

for sid in (SELECTED_STUDIES if EXECUTION_MODE == "BATCH" else SELECTED_STUDIES[:5]):

    sub = source_rows_for_study(sid)

    print(
        f"\nSTUDY {sid}: "
        f"expected {len(sub)} source file(s)"
    )

    cols = [
        c for c in [
            "role",
            "source_file_name",
            "source_file_id",
            "screening_class",
            "source_rank",
            "manifest_missing_local",
        ]
        if c in sub.columns
    ]

    display(sub[cols])

In [ ]:
# ============================================================
# 7. File fingerprints + checkpoints
# ============================================================

def read_json_if_exists(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


# Resume state is mirrored outside the Google Drive mount. Parsed assets stay
# in their original folders. Do not change PIPELINE_VERSION to resume a run.
import tempfile

_resume_project_key = hashlib.sha256(PROJECT_ROOT_DISPLAY.encode("utf-8")).hexdigest()[:16]
LOCAL_RESUME_DIR = (
    Path(os.environ.get("LOCALAPPDATA") or (Path.home() / ".cache"))
    / "SynerPlant" / "extractor_resume" / _resume_project_key / PIPELINE_VERSION
)


def _resume_local_path(path: Path) -> Path:
    # Hashed filenames avoid duplicating deep corpus paths on Windows.
    relative = Path(path).relative_to(PROJECT_ROOT).as_posix()
    key = hashlib.sha256(relative.encode("utf-8")).hexdigest()
    return LOCAL_RESUME_DIR / key[:2] / (key[2:] + Path(path).suffix)


def _atomic_write_text(path: Path, text: str):
    """Never truncate the last saved checkpoint/report in place."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(
            mode="w", encoding="utf-8", newline="", dir=path.parent,
            prefix=".resume_", suffix=".tmp", delete=False,
        ) as f:
            temp_path = Path(f.name)
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, path)
    finally:
        if temp_path is not None:
            try:
                temp_path.unlink(missing_ok=True)
            except OSError:
                pass


def _write_resume_text(path: Path, text: str):
    # Local commit first: a Drive disconnect must not erase saved progress.
    local_path = _resume_local_path(path)
    local_error = None
    try:
        _atomic_write_text(local_path, text)
    except OSError as e:
        local_error = e
    try:
        _atomic_write_text(path, text)
    except OSError:
        if local_error is not None:
            raise
        print(f"RESUME WARNING: Drive copy unavailable; local state saved: {local_path}")
    else:
        if local_error is not None:
            print(f"RESUME WARNING: local backup unavailable ({local_error!r}); saved: {path}")


def _resume_timestamp(record: Dict[str, Any]) -> float:
    value = record.get("_resume_saved_at_utc") or record.get("completed_at_utc")
    try:
        dt = datetime.fromisoformat(str(value))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.timestamp()
    except (TypeError, ValueError, OverflowError, OSError):
        return 0.0


def read_checkpoint(path: Path) -> Optional[Dict[str, Any]]:
    local_path = _resume_local_path(path)
    candidates = []
    # Prefer the local copy on equal timestamps; accept existing V4.2 checkpoints.
    for candidate in (local_path, path):
        try:
            cp = read_json_if_exists(candidate)
        except OSError:
            cp = None
        if isinstance(cp, dict) and cp:
            candidates.append((candidate, cp))
    if not candidates:
        return None
    selected_path, cp = max(candidates, key=lambda item: _resume_timestamp(item[1]))
    if selected_path != local_path:
        # Also back up checkpoints produced by the original notebook.
        try:
            _atomic_write_text(local_path, json.dumps(cp, ensure_ascii=False, indent=2))
        except OSError as e:
            print(f"RESUME WARNING: cannot back up {path.name}: {e!r}")
    return cp


def write_checkpoint(path: Path, checkpoint: Dict[str, Any]):
    payload = {**checkpoint, "_resume_saved_at_utc": datetime.now(timezone.utc).isoformat()}
    _write_resume_text(path, json.dumps(payload, ensure_ascii=False, indent=2))


def source_signatures_match(old: Dict[str, Any], current: Dict[str, Any]) -> bool:
    if not isinstance(old, dict) or not isinstance(current, dict):
        return False
    identity = ("source_file_name", "source_file_id", "size_bytes")
    if any(k not in old or k not in current or old[k] != current[k] for k in identity):
        return False
    if old.get("sha256") and current.get("sha256"):
        # A timestamp-only change is not a content change when SHA-256 agrees.
        return old["sha256"] == current["sha256"]
    # Without two hashes, retain the original conservative comparison.
    return old == current


def study_signatures_match(old: Any, current: List[Dict[str, Any]]) -> bool:
    return (
        isinstance(old, list) and len(old) == len(current)
        and all(source_signatures_match(a, b) for a, b in zip(old, current))
    )


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def load_hash_cache() -> pd.DataFrame:
    for cache_path in (HASH_CACHE_PATH, _resume_local_path(HASH_CACHE_PATH)):
        try:
            cache = pd.read_csv(cache_path, dtype=str).fillna("")
            if {"path", "size_bytes", "modified_time_ns", "sha256"}.issubset(cache.columns):
                return cache
        except (OSError, ValueError, pd.errors.ParserError):
            pass
    return pd.DataFrame(columns=["path", "size_bytes", "modified_time_ns", "sha256"])


def cached_sha256(path: Path) -> str:
    if HASH_POLICY.upper() == "NONE":
        return ""
    st = path.stat()
    cache = load_hash_cache()
    key = str(path)
    hit = cache[(cache["path"].eq(key)) & cache["size_bytes"].eq(str(st.st_size)) & cache["modified_time_ns"].eq(str(st.st_mtime_ns))]
    if HASH_POLICY.upper() == "ONCE" and not hit.empty:
        return hit.iloc[-1]["sha256"]
    digest = sha256_file(path)
    new = pd.DataFrame([{ "path": key, "size_bytes": str(st.st_size), "modified_time_ns": str(st.st_mtime_ns), "sha256": digest }])
    cache = pd.concat([cache, new], ignore_index=True).drop_duplicates(["path", "size_bytes", "modified_time_ns"], keep="last")
    _write_resume_text(HASH_CACHE_PATH, cache.to_csv(index=False))
    return digest


def source_signature_from_row(row: Dict[str, Any]) -> Dict[str, Any]:
    path = Path(row["source_path"]) if row.get("source_path") else None
    sig = {
        "source_file_name": clean_str(row.get("source_file_name")),
        "source_file_id": clean_str(row.get("source_file_id")),
        "size_bytes": int(row.get("size_bytes") or 0),
        "modified_time_ns": int(row.get("modified_time_ns") or 0),
    }
    if path and path.exists() and HASH_POLICY.upper() != "NONE":
        sig["sha256"] = cached_sha256(path)
    return sig


def study_current_signatures(study_id: str) -> List[Dict[str, Any]]:
    return [source_signature_from_row(r) for r in source_rows_for_study(study_id).to_dict("records")]


def study_checkpoint_is_current(study_id: str) -> bool:
    if FORCE_REPROCESS:
        return False
    cp = read_checkpoint(CHECKPOINT_DIR / f"{study_id}__STUDY_COMPLETE.json")
    return bool(cp and study_signatures_match(cp.get("source_signatures"), study_current_signatures(study_id)))


In [ ]:
# 8. Lazy converters: no model initialization is needed for saved-output upgrades.
def build_docling_converters():
    pdf_options = PdfPipelineOptions()
    pdf_options.generate_page_images = False
    pdf_options.generate_picture_images = True
    pdf_options.images_scale = IMAGE_SCALE
    pdf_options.do_ocr = DO_OCR
    pdf_options.do_table_structure = True
    
    if hasattr(pdf_options, "generate_table_images"):
        pdf_options.generate_table_images = False
    
    if TABLE_MODE_ACCURATE:
        try:
            pdf_options.table_structure_options.mode = TableFormerMode.ACCURATE
        except Exception as e:
            print("Table ACCURATE mode not available:", repr(e))
    
    if hasattr(pdf_options, "enable_remote_services"):
        pdf_options.enable_remote_services = False
    if hasattr(pdf_options, "do_picture_description"):
        pdf_options.do_picture_description = False
    
    # Heavy chart extraction is optional. The first pass remains deterministic/resource-conscious.
    if hasattr(pdf_options, "do_chart_extraction"):
        pdf_options.do_chart_extraction = bool(ENABLE_DOCLING_CHART_EXTRACTION)
        print("Docling chart extraction:", pdf_options.do_chart_extraction)
    else:
        print("Docling chart extraction option not available in this installed version.")
    
    pdf_converter = DocumentConverter(
        allowed_formats=[InputFormat.PDF],
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)},
    )
    generic_converter = DocumentConverter()
    
    return pdf_converter, generic_converter

class LazyConverter:
    def __init__(self, kind):
        self.kind = kind
        self.converter = None
    def convert(self, path):
        if EXECUTION_MODE != "BATCH":
            raise RuntimeError("Docling conversion is disabled outside BATCH mode")
        if self.converter is None:
            pdf, generic = build_docling_converters()
            PDF_CONVERTER.converter, GENERIC_CONVERTER.converter = pdf, generic
        return self.converter.convert(path)
PDF_CONVERTER = LazyConverter("pdf")
GENERIC_CONVERTER = LazyConverter("docx")
print("Converters are lazy; maintenance does not invoke OCR or model downloads.")


In [ ]:
# ============================================================
# 9. Utilities, type sniffing, provenance and objective-driven candidate mining
# ============================================================

QUANT_CAPTION_RE = re.compile(
    r"(?i)\b(figure|fig\.?|yield|biomass|weight|height|length|area|content|concentration|activity|"
    r"chlorophyll|carotenoid|photosynth|fluorescen|fv/fm|npq|proline|mda|catalase|cat\b|sod\b|pod\b|"
    r"enzyme|growth|root|shoot|fruit|disease|severity|incidence|AUDPC|lesion|mortality|survival|"
    r"pathogen|CFU|colony|viral|titer|gall|nematode|relative water|rwc|electrolyte|conductance|stomatal|"
    r"gas exchange|metabol|hormone|ion|nutrient|expression|transcript|protein)\b"
)
QUANT_TEXT_RE = re.compile(
    r"(?i)(?:[-+]?\d+(?:\.\d+)?\s*(?:±|%|mg|g|kg|µg|ug|ng|CFU|cfu|spores?|conidia|zoospores?|eggs?|J2|"
    r"mmol|µmol|umol|mol|cm|mm|mM|µM|MPa|dS\s*m[-−]?1|day|days|d\b|dpi|dai|dat|h\b|hours|min|°C)|p\s*[<=>]\s*0?\.\d+)"
)
REPLICATE_RE = re.compile(r"(?i)\b(n\s*=|replicat|plants? per treatment|biological replic|technical replic|independent experiment|sample size|experimental unit)\b")
STATS_RE = re.compile(r"(?i)\b(ANOVA|factorial|two[- ]way|three[- ]way|interaction|mixed model|linear model|GLM|Tukey|Duncan|LSD|Bonferroni|Dunnett|Kruskal|Mann[- ]Whitney|p\s*[<=>]|standard deviation|standard error|confidence interval|mean\s*±)\b")
METHOD_RE = re.compile(r"(?i)\b(materials? and methods?|experimental design|growth conditions?|treatment|inoculat|infection|pathogen|disease assessment|assay|measurement|determination|analysis was performed|instrument|spectro|chromat|PCR|qPCR|sequenc|CFU|plating)\b")
RESULT_RE = re.compile(r"(?i)\b(results?|significantly|increased|decreased|higher|lower|compared with|compared to|effect of|disease severity|disease incidence|AUDPC|lesion|pathogen load)\b")
STRESS_RE = re.compile(r"(?i)\b(drought|water deficit|salin|NaCl|salt stress|heat|high temperature|cold|chill|low temperature|waterlog|hypoxi|osmotic|nutrient|heavy metal|toxic|UV|radiation|disease|pathogen|infection|fung|bacter|virus|viroid|oomyc|nemat|insect|herbivor|Fusarium|Verticillium|Alternaria|Botrytis|Phytophthora|Pythium|Ralstonia|Pseudomonas|Xanthomonas|Clavibacter|TYLCV|TMV|ToMV|blight|wilt|rot|mildew|mosaic|speck|spot|root-knot)\b")
BIOSTIM_RE = re.compile(r"(?i)\b(biostimul|seaweed|algae|extract|PGPR|rhizobacter|mycorrh|AMF|Trichoderma|Bacillus|Pseudomonas|protein hydrolysate|amino acid|humic|fulvic|chitosan|silicon|selenium|biochar|compost|nano|inocul|microbial|commercial product|foliar spray|seed priming|biopriming)\b")
BIOTIC_DETAIL_RE = re.compile(r"(?i)\b(inocul|conidia|spore|zoospore|mycel|CFU|colony|isolate|strain|race|pathovar|forma specialis|f\.\s*sp\.|dpi|days? post[- ]inoculation|AUDPC|disease index|severity scale|incidence|lesion|viral titer|galling|egg mass|J2)\b")
NUMERIC_TOKEN_RE = re.compile(r"^[+-]?(?:\d+(?:\.\d+)?|\.\d+)$")

# Candidate mining is recall-oriented. These are hints, never final screening labels.
_STEMS = {
    "replicat", "salin", "waterlog", "hypoxi", "chill", "fung", "bacter", "oomyc", "nemat",
    "biostimul", "rhizobacter", "mycorrh", "inocul", "inoculat", "spectro", "chromat", "sequenc",
    "photosynth", "fluorescen", "metabol", "stomatal", "herbivor",
}
def _expand_candidate_stems(pattern):
    text = pattern.pattern
    for stem in sorted(_STEMS, key=len, reverse=True):
        text = re.sub(r"(?<=[(|])" + re.escape(stem) + r"(?=[|)])", lambda m: m.group(0) + r"\w*", text)
    return re.compile(text, pattern.flags)
for _regex_name in ["QUANT_CAPTION_RE", "REPLICATE_RE", "METHOD_RE", "STRESS_RE", "BIOSTIM_RE", "BIOTIC_DETAIL_RE"]:
    globals()[_regex_name] = _expand_candidate_stems(globals()[_regex_name])

def safe_name(s: str, max_len: int = 120) -> str:
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")
    return s[:max_len] or "file"


def relative_to_project(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)


def source_local_id(path: Path) -> str:
    return "local_" + hashlib.sha1(relative_to_project(path).encode("utf-8")).hexdigest()[:12]


def sniff_file_type(path: Path, declared_mime: str = "") -> str:
    with path.open("rb") as f:
        head = f.read(32)
    if head.startswith(b"%PDF"):
        return "pdf"
    if head.startswith(bytes.fromhex("D0CF11E0A1B11AE1")):
        return "xls" if path.suffix.lower() == ".xls" else "doc_legacy"
    if head.startswith(b"PK\x03\x04"):
        try:
            with zipfile.ZipFile(path) as z:
                names = set(z.namelist())
            if "xl/workbook.xml" in names:
                return "xlsx"
            if "word/document.xml" in names:
                return "docx"
        except Exception:
            pass
        return "zip_ooxml"
    ext = path.suffix.lower()
    if ext in {".csv", ".tsv"}:
        return "csv"
    if ext == ".xlsx":
        return "xlsx"
    if ext == ".xls":
        return "xls"
    if ext == ".docx":
        return "docx"
    if ext == ".doc":
        return "doc_legacy"
    if ext == ".pdf":
        return "pdf"
    return "unknown"


def prov_fields(item, doc) -> Dict[str, Any]:
    out = {"page_no": None, "bbox_l": None, "bbox_t": None, "bbox_r": None, "bbox_b": None, "bbox_origin": None}
    try:
        if item.prov:
            p = item.prov[0]
            b = p.bbox
            out.update({
                "page_no": int(p.page_no), "bbox_l": float(b.l), "bbox_t": float(b.t),
                "bbox_r": float(b.r), "bbox_b": float(b.b), "bbox_origin": str(b.coord_origin),
            })
    except Exception:
        pass
    return out


def caption_text(item, doc) -> str:
    try:
        return clean_str(item.caption_text(doc))
    except Exception:
        return ""


def model_dump_safe(obj) -> Any:
    if obj is None:
        return None
    for method in ["model_dump", "dict"]:
        if hasattr(obj, method):
            try:
                return getattr(obj, method)(mode="json") if method == "model_dump" else getattr(obj, method)()
            except Exception:
                try:
                    return getattr(obj, method)()
                except Exception:
                    pass
    return str(obj)


def candidate_flags(text: str) -> Dict[str, bool]:
    return {
        "is_quantitative": bool(QUANT_TEXT_RE.search(text)),
        "is_method_candidate": bool(METHOD_RE.search(text)),
        "is_result_candidate": bool(RESULT_RE.search(text)),
        "is_stress_candidate": bool(STRESS_RE.search(text)),
        "is_biostimulant_candidate": bool(BIOSTIM_RE.search(text)),
        "is_biotic_detail_candidate": bool(BIOTIC_DETAIL_RE.search(text)),
        "is_replication_candidate": bool(REPLICATE_RE.search(text)),
        "is_statistics_candidate": bool(STATS_RE.search(text)),
    }


def extract_asset_refs(text: str, kind: str) -> List[str]:
    pat = r"(?i)\bTable\s+([A-Za-z]?\d+[A-Za-z]?)" if kind == "table" else r"(?i)\b(?:Figure|Fig\.)\s+([A-Za-z]?\d+[A-Za-z]?)"
    return sorted(set(x.upper() for x in re.findall(pat, text)))


def image_geometry_features(img: Image.Image) -> Dict[str, Any]:
    arr = np.array(img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)

    min_len = max(20, int(min(gray.shape[:2]) * 0.25))

    lines = cv2.HoughLinesP(
        edges,
        1,
        np.pi / 180,
        threshold=60,
        minLineLength=min_len,
        maxLineGap=8,
    )

    horizontal = vertical = 0

    if lines is not None:
        lines = np.asarray(lines).reshape(-1, 4)

        for l in lines:
            x1, y1, x2, y2 = map(int, l)
            dx, dy = abs(x2 - x1), abs(y2 - y1)

            if dx > 3 * max(dy, 1):
                horizontal += 1
            elif dy > 3 * max(dx, 1):
                vertical += 1

    return {
        "image_width": int(gray.shape[1]),
        "image_height": int(gray.shape[0]),
        "hough_horizontal_lines": int(horizontal),
        "hough_vertical_lines": int(vertical),
        "chart_geometry_hint": bool(horizontal >= 1 and vertical >= 1),
    }

# A literal n= does not end with a word character; no trailing boundary here.
REPLICATE_RE = re.compile(r"(?i)(?:\bn\s*=\s*\d+|\breplicat\w*|\bplants? per treatment\b|\bbiological replic\w*|\btechnical replic\w*|\bindependent experiment\w*|\bsample size\b|\bexperimental unit\w*)")


In [ ]:
# ============================================================
# 10. Page-aware text inventory and semantic candidate chunks
# ============================================================

def build_text_inventory(doc) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = []
    current_section = ""
    for idx, (item, level) in enumerate(doc.iterate_items(), 1):
        txt = clean_str(getattr(item, "text", ""))
        if not txt:
            continue
        item_type = type(item).__name__
        label = clean_str(getattr(item, "label", ""))
        low = (item_type + " " + label).lower()
        if "section" in low or "heading" in low or "title" in low:
            current_section = txt[:300]
        row = {
            "text_item_id": idx, "item_type": item_type, "label": label, "level": level,
            "section": current_section, "text": txt, **prov_fields(item, doc), **candidate_flags(txt),
        }
        rows.append(row)

    candidates = [r for r in rows if any(v for k, v in r.items() if k.startswith("is_") and isinstance(v, bool))]
    return rows, candidates


def write_text_chunks(rows: List[Dict[str, Any]], path: Path, max_chars: int = 1800):
    chunks = []
    buf, meta = [], None
    def flush():
        nonlocal buf, meta
        if not buf:
            return
        chunks.append({**meta, "text": " ".join(buf)})
        buf, meta = [], None

    for r in rows:
        key = (r.get("section"), r.get("page_no"))
        text = r.get("text", "")
        if meta is None:
            meta = {"section": r.get("section"), "page_no": r.get("page_no"), "first_text_item_id": r.get("text_item_id"), "last_text_item_id": r.get("text_item_id")}
        would = len(" ".join(buf)) + len(text) + 1
        if (meta.get("section"), meta.get("page_no")) != key or would > max_chars:
            flush()
            meta = {"section": r.get("section"), "page_no": r.get("page_no"), "first_text_item_id": r.get("text_item_id"), "last_text_item_id": r.get("text_item_id")}
        buf.append(text)
        meta["last_text_item_id"] = r.get("text_item_id")
    flush()

    with path.open("w", encoding="utf-8") as f:
        for i, ch in enumerate(chunks, 1):
            ch["chunk_id"] = i
            f.write(json.dumps(ch, ensure_ascii=False) + "\n")


In [ ]:
# ============================================================
# 11. Figure/vector helpers and conservative bar digitization
# ============================================================

def fitz_words_in_rect(pdf_path: Path, page_no_1based: int, rect: fitz.Rect) -> List[Dict[str, Any]]:
    out = []
    with fitz.open(pdf_path) as pdf:
        if page_no_1based < 1 or page_no_1based > len(pdf):
            return out
        page = pdf[page_no_1based - 1]
        for w in page.get_text("words"):
            x0, y0, x1, y1, text, block, line, word = w[:8]
            wr = fitz.Rect(x0, y0, x1, y1)
            if rect.intersects(wr):
                out.append({"text": clean_str(text), "x0": float(x0), "y0": float(y0), "x1": float(x1), "y1": float(y1)})
    return out


def docling_bbox_to_fitz(pdf_path: Path, page_no_1based: int, prov_bbox) -> fitz.Rect:
    with fitz.open(pdf_path) as pdf:
        page = pdf[page_no_1based - 1]
        h = page.rect.height
    try:
        b = prov_bbox.to_top_left_origin(page_height=h)
        return fitz.Rect(float(b.l), float(b.t), float(b.r), float(b.b))
    except Exception:
        return fitz.Rect(float(prov_bbox.l), float(prov_bbox.t), float(prov_bbox.r), float(prov_bbox.b))


def render_pdf_rect(pdf_path: Path, page_no_1based: int, rect: fitz.Rect, out_path: Path, scale: float = FIGURE_RENDER_SCALE) -> bool:
    try:
        with fitz.open(pdf_path) as pdf:
            page = pdf[page_no_1based - 1]
            clip = rect & page.rect
            pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), clip=clip, alpha=False)
            pix.save(str(out_path))
        return True
    except Exception:
        return False


def vector_primitives_in_rect(pdf_path: Path, page_no_1based: int, rect: fitz.Rect) -> List[Dict[str, Any]]:
    primitives = []
    with fitz.open(pdf_path) as pdf:
        page = pdf[page_no_1based - 1]
        for d in page.get_drawings():
            dr = d.get("rect")
            if dr is None or not rect.intersects(dr):
                continue
            item_list = []
            for item in d.get("items", []):
                kind = item[0]
                vals = []
                for obj in item[1:]:
                    if hasattr(obj, "x") and hasattr(obj, "y"):
                        vals.append([float(obj.x), float(obj.y)])
                    elif isinstance(obj, fitz.Rect):
                        vals.append([float(obj.x0), float(obj.y0), float(obj.x1), float(obj.y1)])
                    else:
                        try:
                            vals.append(float(obj))
                        except Exception:
                            vals.append(str(obj))
                item_list.append([kind, *vals])
            primitives.append({
                "type": d.get("type"), "fill": str(d.get("fill")), "color": str(d.get("color")),
                "width": d.get("width"), "rect": [float(dr.x0), float(dr.y0), float(dr.x1), float(dr.y1)], "items": item_list,
            })
    return primitives


def _line_segments_from_drawings(drawings: List[Dict[str, Any]]) -> List[Tuple[float, float, float, float]]:
    segs = []
    for d in drawings:
        for item in d.get("items", []):
            if not item or item[0] != "l" or len(item) < 3:
                continue
            p1, p2 = item[1], item[2]
            if isinstance(p1, list) and isinstance(p2, list) and len(p1) == 2 and len(p2) == 2:
                segs.append((p1[0], p1[1], p2[0], p2[1]))
    return segs


def try_vector_bar_digitization(pdf_path: Path, page_no: int, fig_rect: fitz.Rect,
                                words: List[Dict[str, Any]], drawings: List[Dict[str, Any]]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    report = {"status": "NOT_DIGITIZED", "reason": "", "calibration_r2": None}
    segs = _line_segments_from_drawings(drawings)
    if not segs:
        report["reason"] = "no_vector_line_segments"
        return pd.DataFrame(), report

    fw, fh = fig_rect.width, fig_rect.height
    verticals, horizontals = [], []
    for x1, y1, x2, y2 in segs:
        dx, dy = abs(x2 - x1), abs(y2 - y1)
        if dy > 0.30 * fh and dx < 0.03 * fw:
            verticals.append((x1, y1, x2, y2, dy))
        if dx > 0.35 * fw and dy < 0.03 * fh:
            horizontals.append((x1, y1, x2, y2, dx))
    if not verticals or not horizontals:
        report["reason"] = "axes_not_detected"
        return pd.DataFrame(), report

    y_axis = sorted(verticals, key=lambda z: (abs(z[0] - fig_rect.x0), -z[4]))[0]
    x_axis = sorted(horizontals, key=lambda z: (abs(z[1] - fig_rect.y1), -z[4]))[0]
    y_axis_x = (y_axis[0] + y_axis[2]) / 2
    baseline_y = (x_axis[1] + x_axis[3]) / 2

    ticks = []
    for w in words:
        txt = w["text"].replace("−", "-").replace(",", "")
        if not NUMERIC_TOKEN_RE.match(txt):
            continue
        xc = (w["x0"] + w["x1"]) / 2
        yc = (w["y0"] + w["y1"]) / 2
        if fig_rect.x0 - 2 <= xc <= y_axis_x + 0.03 * fw and fig_rect.y0 <= yc <= fig_rect.y1:
            try:
                ticks.append((yc, float(txt)))
            except Exception:
                pass
    dedup = []
    for y, val in sorted(ticks):
        if not dedup or abs(y - dedup[-1][0]) > 1.5 or abs(val - dedup[-1][1]) > 1e-12:
            dedup.append((y, val))
    ticks = dedup
    if len(ticks) < 2:
        report["reason"] = "insufficient_numeric_y_ticks"
        return pd.DataFrame(), report

    ys = np.array([y for y, _ in ticks], dtype=float)
    vals = np.array([v for _, v in ticks], dtype=float)
    coef = np.polyfit(ys, vals, 1)
    pred = np.polyval(coef, ys)
    ss_res = float(np.sum((vals - pred) ** 2))
    ss_tot = float(np.sum((vals - np.mean(vals)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 1.0
    report.update({"calibration_r2": r2, "y_axis_x": y_axis_x, "baseline_y": baseline_y, "tick_count": len(ticks)})
    if r2 < MIN_BAR_DIGITIZATION_R2:
        report["reason"] = "weak_y_axis_calibration"
        return pd.DataFrame(), report

    bars = []
    for d in drawings:
        dr = d.get("rect")
        if not dr or len(dr) != 4:
            continue
        x0, y0, x1, y1 = dr
        width, height = x1 - x0, y1 - y0
        if width <= 2 or height <= 3 or x0 < y_axis_x - 1 or x1 > fig_rect.x1 + 1:
            continue
        if y0 < fig_rect.y0 - 1 or y1 > fig_rect.y1 + 1:
            continue
        if abs(y1 - baseline_y) > max(8.0, 0.06 * fh):
            continue
        if width > 0.30 * fw or height < 0.04 * fh:
            continue
        bars.append((x0, y0, x1, y1))

    uniq = []
    for b in sorted(bars, key=lambda z: (z[0], z[1], z[2], z[3])):
        if not uniq or max(abs(b[i] - uniq[-1][i]) for i in range(4)) > 1.0:
            uniq.append(b)
    bars = uniq
    if not (2 <= len(bars) <= 40):
        report["reason"] = f"implausible_bar_count_{len(bars)}"
        return pd.DataFrame(), report

    rows = []
    for i, (x0, y0, x1, y1) in enumerate(sorted(bars, key=lambda z: (z[0] + z[2]) / 2), 1):
        xc = (x0 + x1) / 2
        near = []
        for w in words:
            wx = (w["x0"] + w["x1"]) / 2
            wy = (w["y0"] + w["y1"]) / 2
            if abs(wx - xc) <= max(20, 0.08 * fw) and baseline_y <= wy <= fig_rect.y1 + 25:
                near.append(w["text"])
        rows.append({
            "bar_index": i, "x_center_pdf": xc, "bar_top_y_pdf": y0,
            "digitized_value_candidate": float(np.polyval(coef, y0)),
            "x_text_nearby_raw": " ".join(near).strip(),
            "verified": False, "uncertainty_extracted": False,
        })
    report.update({"status": "AUTO_VECTOR_BAR_CANDIDATE", "reason": "conservative_geometry_pass", "bar_count": len(rows)})
    return pd.DataFrame(rows), report


In [ ]:
# ============================================================
# 12. Parse PDF: text + tables + all figures + quantitative queues
# ============================================================

def nearby_text_for_bbox(text_rows, page_no, rect, margin=80.0, page_height=None):
    parts = []
    expanded = fitz.Rect(rect.x0-margin, rect.y0-margin, rect.x1+margin, rect.y1+margin)
    for row in text_rows:
        if row.get("page_no") != page_no or row.get("bbox_l") is None:
            continue
        l, t, r, b = [float(row[k]) for k in ("bbox_l", "bbox_t", "bbox_r", "bbox_b")]
        origin = str(row.get("bbox_origin", "")).upper()
        if "BOTTOMLEFT" in origin or "BOTTOM_LEFT" in origin:
            if page_height is None:
                continue
            t, b = page_height-t, page_height-b
        rr = fitz.Rect(min(l,r), min(t,b), max(l,r), max(t,b))
        if expanded.intersects(rr):
            parts.append(row.get("text", ""))
    return " ".join(parts)[:6000]


def parse_pdf_source(pdf_path: Path, out_dir: Path, source_meta: Dict[str, Any]) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = out_dir / "tables"
    figures_dir = out_dir / "figures"
    tables_dir.mkdir(exist_ok=True)
    figures_dir.mkdir(exist_ok=True)
    started = time.time()

    conv = PDF_CONVERTER.convert(pdf_path)
    doc = conv.document
    conversion_status = str(getattr(conv, "status", "UNKNOWN"))

    md_path = out_dir / "document.md"
    txt_path = out_dir / "document.txt"
    json_path = out_dir / "docling_document.json"
    doc.save_as_markdown(md_path, image_mode=ImageRefMode.PLACEHOLDER)
    doc.save_as_markdown(txt_path, image_mode=ImageRefMode.PLACEHOLDER, strict_text=True)
    doc.save_as_json(json_path, image_mode=ImageRefMode.PLACEHOLDER, ensure_ascii=False)
    markdown_text = md_path.read_text(encoding="utf-8", errors="replace")

    with fitz.open(pdf_path) as pdf:
        pdf_pages = len(pdf)
    parsed_pages = len(getattr(doc, "pages", {}) or {})

    text_rows, semantic_candidates = build_text_inventory(doc)
    pd.DataFrame(text_rows).to_csv(out_dir / "text_items.csv", index=False)
    pd.DataFrame(semantic_candidates).to_csv(out_dir / "semantic_text_candidates.csv", index=False)
    write_text_chunks(text_rows, out_dir / "text_chunks.jsonl")

    # Tables: CSV + image + provenance.
    table_rows, table_failures = [], []
    for i, table in enumerate(doc.tables, 1):
        info = {"table_index": i, "caption": caption_text(table, doc), **prov_fields(table, doc), "export_status": "OK"}
        try:
            df = table.export_to_dataframe(doc=doc)
            csv_path = tables_dir / f"table_{i:03d}.csv"
            df.to_csv(csv_path, index=False)
            info.update({"csv_file": csv_path.name, "n_rows": int(df.shape[0]), "n_cols": int(df.shape[1]), "nonempty_cells": int(df.notna().sum().sum())})
            try:
                info["numeric_cell_count"] = sum(int(pd.to_numeric(df.iloc[:, j], errors="coerce").notna().sum()) for j in range(df.shape[1]))
            except Exception as metric_error:
                info["numeric_cell_count"] = None
                info["numeric_count_warning"] = repr(metric_error)
            try:
                table_img = table.get_image(doc)
                if table_img is not None:
                    img_path = tables_dir / f"table_{i:03d}.png"
                    table_img.save(img_path)
                    info["table_image_file"] = img_path.name
                    info["table_image_status"] = "OK"
                else:
                    info["table_image_status"] = "MISSING"
            except Exception as e:
                info["table_image_status"] = "FAILED"
                info["table_image_error"] = repr(e)
        except Exception as e:
            info["export_status"] = "FAILED"
            info["error"] = repr(e)
            table_failures.append(i)
        table_rows.append(info)
    pd.DataFrame(table_rows).to_csv(out_dir / "table_inventory.csv", index=False)

    # Figures: keep every figure; never discard non-quantitative pictures because they can encode phenotype/method context.
    fig_rows, digitization_queue = [], []
    for i, pic in enumerate(doc.pictures, 1):
        info = {
            "figure_index": i, "caption": caption_text(pic, doc), **prov_fields(pic, doc),
            "image_status": "MISSING", "likely_quantitative": False,
            "digitization_status": "NOT_REQUIRED_OR_UNKNOWN", "docling_picture_meta": model_dump_safe(getattr(pic, "meta", None)),
        }
        img = None
        try:
            img = pic.get_image(doc)
        except Exception as e:
            info["get_image_error"] = repr(e)

        fig_rect, words, drawings = None, [], []
        try:
            if pic.prov:
                p = pic.prov[0]
                fig_rect = docling_bbox_to_fitz(pdf_path, int(p.page_no), p.bbox)
                words = fitz_words_in_rect(pdf_path, int(p.page_no), fig_rect)
                drawings = vector_primitives_in_rect(pdf_path, int(p.page_no), fig_rect)
        except Exception as e:
            info["vector_capture_error"] = repr(e)

        fig_path = figures_dir / f"figure_{i:03d}.png"
        if img is not None:
            img.save(fig_path)
            info["image_file"] = fig_path.name
            info["image_status"] = "OK"
            info.update(image_geometry_features(img))
        elif fig_rect is not None and info.get("page_no"):
            if render_pdf_rect(pdf_path, int(info["page_no"]), fig_rect, fig_path):
                info["image_file"] = fig_path.name
                info["image_status"] = "OK_RENDERED_FROM_PAGE"
                try:
                    with Image.open(fig_path) as rimg:
                        info.update(image_geometry_features(rimg))
                except Exception:
                    pass

        info.setdefault("chart_geometry_hint", False)
        info["native_text_token_count"] = len(words)
        info["vector_drawing_count"] = len(drawings)
        info["native_text_in_bbox"] = " | ".join(w["text"] for w in words)[:6000]
        if fig_rect is not None and info.get("page_no"):
            info["nearby_page_text"] = nearby_text_for_bbox(text_rows, int(info["page_no"]), fig_rect, page_height=float(doc.pages[int(info["page_no"])].size.height))

        if drawings:
            (figures_dir / f"figure_{i:03d}__vector_primitives.json").write_text(json.dumps(drawings, ensure_ascii=False, indent=2), encoding="utf-8")
        if words:
            pd.DataFrame(words).to_csv(figures_dir / f"figure_{i:03d}__native_text.csv", index=False)

        caption_hint = bool(QUANT_CAPTION_RE.search(info.get("caption") or ""))
        numeric_tokens = sum(1 for w in words if re.search(r"\d", w["text"]))
        info["likely_quantitative"] = bool(caption_hint or (info.get("chart_geometry_hint") and numeric_tokens >= 2))

        if info["likely_quantitative"]:
            info["digitization_status"] = "NEEDS_DIGITIZATION_OR_DUPLICATE_CHECK"
            class_hint = str(source_meta.get("screening_class") or "").upper()
            work_priority_hint = "P1_CLASS_A" if class_hint == "A" else "P2_SUPPORTING"
            digit_row = {
                "figure_index": i, "page_no": info.get("page_no"), "caption": info.get("caption"),
                "image_file": info.get("image_file", ""), "reason": "quantitative_figure_candidate",
                "screening_class": class_hint, "work_priority_hint": work_priority_hint,
                "auto_status": "NOT_ATTEMPTED", "verified": False,
            }
            if TRY_VECTOR_BAR_DIGITIZATION and fig_rect is not None and drawings and words:
                try:
                    bars_df, rep = try_vector_bar_digitization(pdf_path, int(info["page_no"]), fig_rect, words, drawings)
                    digit_row.update({
                        "auto_status": rep.get("status"), "auto_reason": rep.get("reason"),
                        "calibration_r2": rep.get("calibration_r2"), "auto_bar_count": rep.get("bar_count"),
                    })
                    (figures_dir / f"figure_{i:03d}__digitization_report.json").write_text(json.dumps(rep, ensure_ascii=False, indent=2), encoding="utf-8")
                    if not bars_df.empty:
                        bars_df.to_csv(figures_dir / f"figure_{i:03d}__auto_vector_bars.csv", index=False)
                        info["digitization_status"] = "AUTO_VECTOR_CANDIDATE_NEEDS_VERIFICATION"
                except Exception as e:
                    digit_row.update({"auto_status": "FAILED", "auto_reason": repr(e)})
            digitization_queue.append(digit_row)
        fig_rows.append(info)

    pd.DataFrame(fig_rows).to_csv(out_dir / "figure_inventory.csv", index=False)
    pd.DataFrame(digitization_queue).to_csv(out_dir / "figure_digitization_queue.csv", index=False)

    table_refs = extract_asset_refs(markdown_text, "table")
    fig_refs = extract_asset_refs(markdown_text, "figure")
    qfig_count = sum(bool(x.get("likely_quantitative")) for x in fig_rows)
    qfig_exported = sum(bool(x.get("likely_quantitative")) and str(x.get("image_status", "")).startswith("OK") for x in fig_rows)

    warnings = []
    if fig_refs and not fig_rows:
        warnings.append("FIGURE_REFERENCES_PRESENT_BUT_NO_FIGURES_DETECTED")
    if table_refs and not table_rows:
        warnings.append("TABLE_REFERENCES_PRESENT_BUT_NO_TABLES_DETECTED")

    qa = {
        "source_file_id": source_meta.get("source_file_id"), "source_file_name": source_meta.get("source_file_name"),
        "source_path": source_meta.get("source_path"), "role": source_meta.get("role"), "parser": "docling_pdf",
        "pipeline_version": PIPELINE_VERSION, "conversion_status": conversion_status, "pdf_pages": pdf_pages, "parsed_pages": parsed_pages,
        "page_coverage": (parsed_pages / pdf_pages) if pdf_pages else None, "text_items": len(text_rows),
        "semantic_text_candidates": len(semantic_candidates), "tables_detected": len(table_rows),
        "tables_exported": sum(x.get("export_status") == "OK" for x in table_rows),
        "figures_detected": len(fig_rows), "figures_exported": sum(str(x.get("image_status", "")).startswith("OK") for x in fig_rows),
        "quantitative_figures_detected": qfig_count, "quantitative_figures_exported": qfig_exported,
        "quantitative_figure_asset_coverage": (qfig_exported / qfig_count) if qfig_count else 1.0,
        "quantitative_figures_pending_reconciliation": qfig_count, "table_references_in_text": table_refs,
        "figure_references_in_text": fig_refs, "table_export_failures": table_failures, "warnings": warnings,
        "elapsed_sec": round(time.time() - started, 2),
    }
    hard_fail = ("partial" in conversion_status.lower() or "failure" in conversion_status.lower()) or (pdf_pages and parsed_pages != pdf_pages) or qa["tables_exported"] != qa["tables_detected"] or qa["figures_exported"] != qa["figures_detected"]
    qa["source_qa_status"] = "FAIL" if hard_fail else ("NEEDS_REVIEW" if warnings else "PASS")
    qa["semantic_extraction_status"] = "PENDING"
    qa["measurement_completeness_status"] = "NOT_YET_MEASURED"

    (out_dir / "source_qa.json").write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    (out_dir / "source_inventory.json").write_text(json.dumps({
        "source": source_meta, "qa": qa, "tables": table_rows, "figures": fig_rows,
        "semantic_text_candidate_count": len(semantic_candidates), "digitization_queue_count": len(digitization_queue),
    }, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return qa


In [ ]:
# 13. Lossless native tabular extraction: stream rows/cells, preserve coordinates.
import codecs
from contextlib import ExitStack
from itertools import zip_longest
import xml.etree.ElementTree as ET

CELL_COLUMNS = ["sheet_name", "row_1based", "col_1based", "a1", "value", "value_cached",
                "is_formula", "formula", "data_type", "number_format", "cached_value_missing",
                "physical_line_start", "physical_line_end"]

def _csv_field_limit():
    limit = sys.maxsize
    while True:
        try:
            csv.field_size_limit(limit)
            return
        except OverflowError:
            limit //= 10


def delimited_format(path, lookup_name=None):
    raw = path.open("rb")
    try:
        sample = raw.read(262144)
    finally:
        raw.close()
    warning = ""
    encoding = DELIMITED_ENCODING_OVERRIDES.get(lookup_name or path.name)
    if not encoding:
        if sample.startswith(codecs.BOM_UTF8):
            encoding = "utf-8-sig"
        elif sample.startswith((codecs.BOM_UTF32_LE, codecs.BOM_UTF32_BE)):
            encoding = "utf-32"
        elif sample.startswith((codecs.BOM_UTF16_LE, codecs.BOM_UTF16_BE)):
            encoding = "utf-16"
        else:
            try:
                codecs.getincrementaldecoder("utf-8")().decode(sample, final=False)
                encoding = "utf-8"
            except UnicodeDecodeError:
                from charset_normalizer import from_bytes
                guess = from_bytes(sample).best()
                encoding = guess.encoding if guess else "latin1"
                warning = "ENCODING_GUESSED_REVIEW_OR_SET_OVERRIDE"
    text = codecs.getincrementaldecoder(encoding)().decode(sample, final=False)
    sep = DELIMITED_SEPARATOR_OVERRIDES.get(lookup_name or path.name)
    if not sep:
        if path.suffix.lower() == ".tsv":
            sep = "\t"
        else:
            try:
                sep = csv.Sniffer().sniff(text, delimiters=",;\t|").delimiter
            except csv.Error:
                # Single-column files are valid; ambiguous multi-column files need review.
                sep = ","
                if any(c in text for c in ";\t|"):
                    warning = (warning + ";" if warning else "") + "DELIMITER_AMBIGUOUS_SET_OVERRIDE"
    if len(sep) != 1:
        raise ValueError("CSV delimiter must be one character")
    return encoding, sep, warning


def iter_delimited_rows(path, encoding, sep):
    _csv_field_limit()
    with path.open("r", encoding=encoding, newline="", errors="strict") as handle:
        reader = csv.reader(handle, delimiter=sep, strict=True)
        previous_line = 0
        for row_no, values in enumerate(reader, 1):
            yield row_no, values, previous_line + 1, reader.line_num
            previous_line = reader.line_num


def read_delimited_robust(path):
    """Compatibility helper: preserve literal strings and blank logical records."""
    encoding, sep, warning = delimited_format(path)
    rows = [r for _, r, _, _ in iter_delimited_rows(path, encoding, sep)]
    width = max((len(r) for r in rows), default=0)
    df = pd.DataFrame([r + [""] * (width - len(r)) for r in rows], dtype=object)
    df.attrs.update(source_encoding=encoding, separator=sep, warning=warning)
    return df, encoding


def xlsx_structures(path):
    """Read structural metadata without expanding a workbook into memory."""
    result = {}
    ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
    with zipfile.ZipFile(path) as z:
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        targets = {r.get("Id"): r.get("Target") for r in rels}
        workbook = ET.fromstring(z.read("xl/workbook.xml"))
        for sheet in workbook.findall("m:sheets/m:sheet", ns):
            rid = sheet.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
            target = targets.get(rid, "")
            name = target.lstrip("/") if target.startswith("/") else "xl/" + target
            if not name or name not in z.namelist():
                continue
            merged, hidden_rows, hidden_cols = [], [], []
            with z.open(name) as xml:
                for _, node in ET.iterparse(xml, events=("end",)):
                    tag = node.tag.rsplit("}", 1)[-1]
                    if tag == "mergeCell":
                        merged.append(node.get("ref"))
                    elif tag == "row" and node.get("hidden") == "1":
                        hidden_rows.append(node.get("r"))
                    elif tag == "col" and node.get("hidden") == "1":
                        hidden_cols.append({"min": node.get("min"), "max": node.get("max")})
                    node.clear()
            result[sheet.get("name")] = {"merged_ranges": merged, "hidden_rows": hidden_rows,
                                         "hidden_columns": hidden_cols, "sheet_state": sheet.get("state", "visible")}
    return result


def _nonempty(value):
    # Whitespace is literal source content, not automatically a missing value.
    return value is not None and value != ""


def parse_tabular_source(path, out_dir, file_type, source_meta):
    out_dir.mkdir(parents=True, exist_ok=True)
    started = time.time()
    warnings, sheets_info = [], []
    stack = ExitStack()
    try:
        if file_type == "xlsx":
            import openpyxl
            formulas = openpyxl.load_workbook(path, read_only=True, data_only=False)
            values = openpyxl.load_workbook(path, read_only=True, data_only=True)
            stack.callback(formulas.close)
            stack.callback(values.close)
            structures = xlsx_structures(path)
            sheet_names = formulas.sheetnames
        elif file_type == "xls":
            import xlrd
            book = xlrd.open_workbook(path, on_demand=True)
            stack.callback(book.release_resources)
            sheet_names = book.sheet_names()
            structures = {}
        else:
            encoding, sep, warning = delimited_format(path, source_meta.get("encoding_lookup_name"))
            if warning:
                warnings.append(warning)
            sheet_names, structures = ["data"], {}
        for idx, name in enumerate(sheet_names, 1):
            stem = f"sheet_{idx:03d}__{safe_name(name)}"
            rect_path = out_dir / (stem + ".csv")
            cells_path = out_dir / (stem + "__cells_long.csv.gz")
            typed_path = out_dir / (stem + "__cells.jsonl.gz")
            nonempty = formula_count = missing_cache = nrows = ncols = exported_cells = 0
            with rect_path.open("w", encoding="utf-8", newline="") as out, \
                 gzip.open(cells_path, "wt", encoding="utf-8", newline="") as longout, \
                 gzip.open(typed_path, "wt", encoding="utf-8") as typedout:
                writer = csv.writer(out)
                longwriter = csv.DictWriter(longout, fieldnames=CELL_COLUMNS)
                longwriter.writeheader()
                def emit(record):
                    nonlocal exported_cells
                    if not EXPORT_CELL_LEVEL_PROVENANCE:
                        return
                    if not STREAM_ALL_CELL_PROVENANCE and exported_cells >= MAX_NONEMPTY_CELLS_FOR_LONG_EXPORT:
                        return
                    longwriter.writerow(record)
                    typedout.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
                    exported_cells += 1
                if file_type == "xlsx":
                    for rnum, (fr, vr) in enumerate(zip_longest(formulas[name].iter_rows(), values[name].iter_rows(), fillvalue=()), 1):
                        nrows = rnum
                        ncols = max(ncols, len(fr), len(vr))
                        row_values = []
                        for cnum, (fc, vc) in enumerate(zip_longest(fr, vr), 1):
                            raw = getattr(fc, "value", None)
                            cached = getattr(vc, "value", None)
                            is_formula = getattr(fc, "data_type", None) == "f"
                            # An absent cached formula result is not converted to a fabricated zero.
                            row_values.append(cached if is_formula and cached is not None else raw)
                            if not _nonempty(raw):
                                continue
                            nonempty += 1
                            formula_count += int(is_formula)
                            missing_cache += int(is_formula and cached is None)
                            emit({"sheet_name": name, "row_1based": rnum, "col_1based": cnum,
                                  "a1": f"{get_column_letter(cnum)}{rnum}", "value": raw,
                                  "value_cached": cached, "is_formula": is_formula,
                                  "formula": raw if is_formula else None,
                                  "data_type": getattr(fc, "data_type", None),
                                  "number_format": getattr(fc, "number_format", None),
                                  "cached_value_missing": bool(is_formula and cached is None)})
                        writer.writerow(row_values)
                elif file_type == "xls":
                    sh = book.sheet_by_name(name)
                    for r in range(sh.nrows):
                        nrows, ncols = r + 1, sh.ncols
                        row_values = []
                        for c in range(sh.ncols):
                            cell = sh.cell(r, c)
                            row_values.append(cell.value)
                            if not _nonempty(cell.value):
                                continue
                            nonempty += 1
                            emit({"sheet_name": name, "row_1based": r+1, "col_1based": c+1,
                                  "a1": f"{get_column_letter(c+1)}{r+1}", "value": cell.value,
                                  "value_cached": cell.value, "is_formula": None, "formula": None,
                                  "data_type": cell.ctype, "number_format": None,
                                  "cached_value_missing": None})
                        writer.writerow(row_values)
                else:
                    layout_path = out_dir / (stem + "__record_layout.jsonl")
                    with layout_path.open("w", encoding="utf-8") as layout:
                        for rnum, row_values, line_start, line_end in iter_delimited_rows(path, encoding, sep):
                            nrows, ncols = rnum, max(ncols, len(row_values))
                            writer.writerow(row_values)  # Retain blank/ragged logical records.
                            layout.write(json.dumps({"row_1based": rnum, "field_count": len(row_values),
                                                     "physical_line_start": line_start, "physical_line_end": line_end}) + "\n")
                            for cnum, raw in enumerate(row_values, 1):
                                if not _nonempty(raw):
                                    continue
                                nonempty += 1
                                emit({"sheet_name": name, "row_1based": rnum, "col_1based": cnum,
                                      "a1": f"{get_column_letter(cnum)}{rnum}", "value": raw,
                                      "value_cached": None, "is_formula": False, "formula": None,
                                      "data_type": "literal_text", "number_format": None,
                                      "cached_value_missing": None,
                                      "physical_line_start": line_start, "physical_line_end": line_end})
            status = "EXPORTED" if exported_cells == nonempty else ("DISABLED" if not EXPORT_CELL_LEVEL_PROVENANCE else "TRUNCATED_BY_USER_LIMIT")
            info = {"sheet_index": idx, "sheet_name": name, "csv_file": rect_path.name,
                    "n_rows": nrows, "n_cols": ncols, "nonempty_cells": nonempty,
                    "cell_provenance_status": status, "cell_provenance_file": cells_path.name,
                    "typed_cell_provenance_file": typed_path.name, "cell_provenance_count": exported_cells,
                    "formula_cells": formula_count, "formula_cached_values_missing": missing_cache,
                    "extraction_method": "SUPPLEMENT_MACHINE_READABLE" if source_meta.get("role") == "SUPPLEMENT" else "TABULAR_NATIVE",
                    **structures.get(name, {})}
            if file_type == "csv":
                info.update(source_encoding=encoding, source_separator=sep,
                            record_layout_file=layout_path.name, a1_semantics="logical CSV record and field; physical lines stored separately")
            if file_type == "xls":
                info["formula_provenance_status"] = "NOT_EXPOSED_BY_XLRD"
            if missing_cache:
                warnings.append(f"FORMULA_CACHED_VALUE_MISSING:{name}:{missing_cache}")
            if status != "EXPORTED":
                warnings.append(f"CELL_PROVENANCE_{status}:{name}")
            sheets_info.append(info)
    finally:
        stack.close()
    total = sum(s["nonempty_cells"] for s in sheets_info)
    if not total:
        warnings.append("EMPTY_TABULAR_SOURCE")
    pd.DataFrame(sheets_info).to_csv(out_dir / "sheet_inventory.csv", index=False)
    qa = {"source_file_id": source_meta.get("source_file_id"), "source_file_name": source_meta.get("source_file_name"),
          "source_path": source_meta.get("source_path"), "role": source_meta.get("role"), "parser": "native_tabular",
          "pipeline_version": PIPELINE_VERSION, "sheets_detected": len(sheets_info), "sheets_exported": len(sheets_info),
          "sheet_coverage": 1.0, "total_nonempty_cells": total, "warnings": warnings,
          "source_qa_status": "NEEDS_REVIEW" if warnings else "PASS", "semantic_extraction_status": "PENDING",
          "elapsed_sec": round(time.time()-started, 2)}
    (out_dir / "source_qa.json").write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    (out_dir / "source_inventory.json").write_text(json.dumps({"source": source_meta, "qa": qa, "sheets": sheets_info}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return qa


In [ ]:
# ============================================================
# 14. DOCX supplements: same evidence philosophy as PDF
# ============================================================

def parse_docx_source(path: Path, out_dir: Path, source_meta: Dict[str, Any]) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = out_dir / "tables"
    figures_dir = out_dir / "figures"
    tables_dir.mkdir(exist_ok=True)
    figures_dir.mkdir(exist_ok=True)
    started = time.time()

    conv = GENERIC_CONVERTER.convert(path)
    doc = conv.document
    conversion_status = str(getattr(conv, "status", "UNKNOWN"))
    doc.save_as_markdown(out_dir / "document.md", image_mode=ImageRefMode.PLACEHOLDER)
    doc.save_as_json(out_dir / "docling_document.json", image_mode=ImageRefMode.PLACEHOLDER, ensure_ascii=False)

    text_rows, semantic_candidates = build_text_inventory(doc)
    pd.DataFrame(text_rows).to_csv(out_dir / "text_items.csv", index=False)
    pd.DataFrame(semantic_candidates).to_csv(out_dir / "semantic_text_candidates.csv", index=False)
    write_text_chunks(text_rows, out_dir / "text_chunks.jsonl")

    table_rows = []
    for i, table in enumerate(doc.tables, 1):
        info = {"table_index": i, "caption": caption_text(table, doc), **prov_fields(table, doc), "export_status": "OK"}
        try:
            df = table.export_to_dataframe(doc=doc)
            csv_path = tables_dir / f"table_{i:03d}.csv"
            df.to_csv(csv_path, index=False)
            info.update({"csv_file": csv_path.name, "n_rows": int(df.shape[0]), "n_cols": int(df.shape[1]), "nonempty_cells": int(df.notna().sum().sum())})
            try:
                timg = table.get_image(doc)
                if timg is not None:
                    timg.save(tables_dir / f"table_{i:03d}.png")
                    info["table_image_status"] = "OK"
            except Exception:
                pass
        except Exception as e:
            info["export_status"] = "FAILED"
            info["error"] = repr(e)
        table_rows.append(info)
    pd.DataFrame(table_rows).to_csv(out_dir / "table_inventory.csv", index=False)

    fig_rows, digitization_queue = [], []
    for i, pic in enumerate(doc.pictures, 1):
        info = {"figure_index": i, "caption": caption_text(pic, doc), **prov_fields(pic, doc), "image_status": "MISSING", "docling_picture_meta": model_dump_safe(getattr(pic, "meta", None))}
        try:
            img = pic.get_image(doc)
            if img is not None:
                f = figures_dir / f"figure_{i:03d}.png"
                img.save(f)
                info["image_file"] = f.name
                info["image_status"] = "OK"
                info.update(image_geometry_features(img))
        except Exception as e:
            info["image_error"] = repr(e)
        info["likely_quantitative"] = bool(QUANT_CAPTION_RE.search(info.get("caption") or "") or info.get("chart_geometry_hint"))
        if info["likely_quantitative"]:
            info["digitization_status"] = "NEEDS_DIGITIZATION_OR_DUPLICATE_CHECK"
            digitization_queue.append({"figure_index": i, "caption": info.get("caption"), "image_file": info.get("image_file", ""), "reason": "quantitative_docx_figure_candidate", "verified": False})
        fig_rows.append(info)
    pd.DataFrame(fig_rows).to_csv(out_dir / "figure_inventory.csv", index=False)
    pd.DataFrame(digitization_queue).to_csv(out_dir / "figure_digitization_queue.csv", index=False)

    qa = {
        "source_file_id": source_meta.get("source_file_id"), "source_file_name": source_meta.get("source_file_name"),
        "role": source_meta.get("role"), "parser": "docling_docx", "conversion_status": conversion_status, "pipeline_version": PIPELINE_VERSION,
        "text_items": len(text_rows), "semantic_text_candidates": len(semantic_candidates),
        "tables_detected": len(table_rows), "tables_exported": sum(x.get("export_status") == "OK" for x in table_rows),
        "figures_detected": len(fig_rows), "figures_exported": sum(x.get("image_status") == "OK" for x in fig_rows),
        "quantitative_figures_detected": sum(bool(x.get("likely_quantitative")) for x in fig_rows),
        "quantitative_figures_exported": sum(bool(x.get("likely_quantitative")) and x.get("image_status") == "OK" for x in fig_rows),
        "source_path": source_meta.get("source_path"),
        "source_qa_status": "PASS", "semantic_extraction_status": "PENDING", "elapsed_sec": round(time.time() - started, 2),
    }
    if qa["tables_exported"] != qa["tables_detected"] or qa["figures_exported"] != qa["figures_detected"]:
        qa["source_qa_status"] = "NEEDS_REVIEW"
    (out_dir / "source_qa.json").write_text(json.dumps(qa, ensure_ascii=False, indent=2), encoding="utf-8")
    (out_dir / "source_inventory.json").write_text(json.dumps({"source": source_meta, "qa": qa, "tables": table_rows, "figures": fig_rows}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return qa


In [ ]:
# 15. Per-source recovery, local work cache, non-destructive publication.
from contextlib import contextmanager


def source_directory(row, study_out):
    path = Path(row["source_path"]) if row.get("source_path") else Path(row.get("source_file_name", "source"))
    sid = clean_str(row.get("source_file_id")) or source_local_id(path)
    role = clean_str(row.get("role"))
    legacy = study_out / f"{role}__{safe_name(path.stem,90)}__{safe_name(sid,16)}"
    compact = study_out / f"{role}__{safe_name(path.stem,35)}__{safe_name(sid,16)}"
    return legacy if legacy.is_dir() else compact


def source_metadata(study_id, row, signature, file_type):
    path = Path(row["source_path"])
    # The entire frozen context survives into handoff: dataset_role, temporal flags, etc.
    context = {k:v for k,v in row.items() if k not in {"source_path", "size_bytes", "modified_time_ns"}}
    return {**context, "study_id": study_id, "source_file_id": clean_str(row.get("source_file_id")) or source_local_id(path),
            "active_file_id": clean_str(row.get("source_file_id")),
            "local_source_id": source_local_id(path), "source_file_name": path.name,
            "source_path": relative_to_project(path), "detected_file_type": file_type,
            "declared_mime_type": row.get("mime_type", ""), "size_bytes": signature["size_bytes"],
            "modified_time_ns": signature["modified_time_ns"], "sha256": signature.get("sha256", ""),
            "pipeline_version": PIPELINE_VERSION, "database_spec": DATABASE_SPEC,
            "extractor_revision": EXTRACTOR_REVISION, "package_versions": DEPENDENCY_VERSIONS,
            "processed_at_utc": datetime.now(timezone.utc).isoformat()}


def try_convert_legacy_doc(path, temp_dir):
    soffice = shutil.which("soffice") or shutil.which("libreoffice")
    if not soffice:
        return None
    temp_dir.mkdir(parents=True, exist_ok=True)
    try:
        subprocess.run([soffice, "--headless", "--convert-to", "docx", "--outdir", str(temp_dir), str(path)],
                       check=True, capture_output=True, text=True, timeout=180)
        candidate = temp_dir / (path.stem + ".docx")
        return candidate if candidate.is_file() else None
    except (subprocess.SubprocessError, OSError):
        return None


def publish_directory(source, destination):
    """Publish only complete output; retain the previous directory in history."""
    incoming = destination.with_name(destination.name + ".incoming")
    if incoming.exists():
        shutil.rmtree(incoming)
    shutil.copytree(source, incoming)
    # Verify copied sizes before the publication marker can be committed.
    for item in source.rglob("*"):
        if item.is_file() and (incoming / item.relative_to(source)).stat().st_size != item.stat().st_size:
            raise IOError(f"Incomplete output copy: {item.name}")
    if destination.exists():
        history = destination.parent / "_history"
        history.mkdir(exist_ok=True)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
        os.replace(destination, history / f"{destination.name}__{stamp}")
    os.replace(incoming, destination)


def _cache_signature_matches(cp, sig):
    return bool(isinstance(cp, dict) and source_signatures_match(cp.get("source_signature", {}), sig))


def base_artifacts_usable(base, qa):
    parser = qa.get("parser", "")
    if parser in {"docling_pdf", "docling_docx"}:
        raw = read_json_if_exists(base / "docling_document.json")
        return isinstance(raw, dict) and all(isinstance(raw.get(k), list) for k in ("texts", "tables", "pictures"))
    if parser == "native_tabular":
        inv = read_json_if_exists(base / "source_inventory.json")
        return isinstance(inv, dict) and isinstance(inv.get("sheets"), list) and all(
            (base / str(sheet.get("csv_file", ""))).is_file() for sheet in inv["sheets"])
    return base.is_dir()


def find_reusable_base(source_out, sig):
    # Dedicated base markers from this version also survive a failed publication.
    for subdir in ("parsed_v4_3", "parsed"):
        base = source_out / subdir
        marker = read_json_if_exists(base / "BASE_COMPLETE.json")
        if _cache_signature_matches(marker, sig) and base_artifacts_usable(base, marker.get("qa", {})):
            qa = dict(marker.get("qa", {}))
            if marker.get("core_settings") and marker["core_settings"] != core_settings_fingerprint():
                qa["needs_core_reparse"] = True
                qa["warnings"] = list(qa.get("warnings") or []) + ["CORE_PARSER_SETTINGS_CHANGED"]
            return base, subdir, qa
    # Compatibility: a completed legacy parser QA + matching source fingerprint.
    cp = read_checkpoint(source_out / "SOURCE_COMPLETE.json") or {}
    if cp.get("_resume_state") == "IN_PROGRESS":
        cp = cp.get("previous_checkpoint") or {}
    qa = cp.get("qa") or {}
    if _cache_signature_matches(cp, sig) and qa.get("parser") in {"docling_pdf", "docling_docx", "native_tabular"}:
        base = source_out / cp.get("base_parsed_subdir", "parsed")
        meta = read_json_if_exists(base / "source_metadata.json") or {}
        if meta.get("sha256") and sig.get("sha256") and meta["sha256"] != sig["sha256"]:
            return None
        if base.is_dir() and base_artifacts_usable(base, qa):
            return base, cp.get("base_parsed_subdir", "parsed"), qa
    # An interrupted legacy export may have saved all Docling detections before failing.
    # Reuse ONLY if the recorded source fingerprint still matches; never trust JSON by filename alone.
    base = source_out / "parsed"
    saved_meta = read_json_if_exists(base / "source_metadata.json") or {}
    metadata_proves_identity = bool(sig.get("sha256") and saved_meta.get("sha256") == sig["sha256"] and
                                    saved_meta.get("source_file_id") == sig.get("source_file_id") and
                                    saved_meta.get("source_file_name") == sig.get("source_file_name"))
    if _cache_signature_matches(cp, sig) or metadata_proves_identity:
        base = source_out / "parsed"
        meta = read_json_if_exists(base / "source_metadata.json") or {}
        raw = read_json_if_exists(base / "docling_document.json")
        expected_type = str(meta.get("detected_file_type", ""))
        source_hash_ok = not meta.get("sha256") or not sig.get("sha256") or meta["sha256"] == sig["sha256"]
        if source_hash_ok and isinstance(raw, dict) and isinstance(raw.get("texts"), list) and expected_type in {"pdf", "docx", "doc_legacy"}:
            recovered = {**qa, "parser": "docling_pdf" if expected_type == "pdf" else "docling_docx",
                         "parsed_pages": len(raw.get("pages") or {}), "conversion_status": "LEGACY_STATUS_UNKNOWN",
                         "warnings": ["RECOVERED_INTERRUPTED_EXPORT_REVIEW_SOURCE_COMPLETENESS"]}
            if expected_type == "pdf":
                relative = meta.get("source_path", "")
                candidate = PROJECT_ROOT / relative if relative else None
                if candidate and candidate.is_file():
                    with fitz.open(candidate) as pdf:
                        recovered["pdf_pages"] = len(pdf)
            return base, "parsed", recovered
    return None


def process_source(study_id, row, study_out):
    source_out = source_directory(row, study_out)
    source_id = clean_str(row.get("source_file_id"))
    status_base = {"study_id": study_id, "source_file_id": source_id,
                   "source_file_name": row.get("source_file_name"), "role": row.get("role")}
    if row.get("manifest_missing_local") or not row.get("source_path"):
        return {**status_base, "source_qa_status": "FAIL", "issue": "MANIFEST_FILE_NOT_FOUND_LOCALLY", "_action": "MISSING_SOURCE"}
    path = Path(row["source_path"])
    sig = source_signature_from_row(row)
    cp_path = source_out / "SOURCE_COMPLETE.json"
    forced = FORCE_REPROCESS or source_id in REPARSE_SOURCE_IDS
    if not forced and checkpoint_matches(cp_path, sig):
        qa = dict((read_checkpoint(cp_path) or {}).get("qa", {}))
        print("  SKIP current:", path.name)
        return {**qa, "_source_folder": source_out.name, "_action": "SKIP_CURRENT", "resumed_from_checkpoint": True}
    ftype = sniff_file_type(path, row.get("mime_type", ""))
    reusable = None if forced else find_reusable_base(source_out, sig)
    # Repair mode NEVER substitutes model conversion for a missing cache.
    if EXECUTION_MODE == "AUDIT_EXISTING":
        action = "REEXPORT_NATIVE" if ftype in {"xlsx", "xls", "csv"} else ("UPGRADE_FROM_CACHE" if reusable else "NEEDS_REPARSE")
        return {**status_base, "source_qa_status": "NOT_RUN", "_action": action, "_source_folder": source_out.name}
    if EXECUTION_MODE != "BATCH" and ftype not in {"xlsx", "xls", "csv"} and reusable is None:
        return {**status_base, "source_qa_status": "FAIL", "issue": "NO_TRUSTED_SAVED_DOCUMENT_FOR_LIGHT_UPGRADE", "_action": "NEEDS_REPARSE", "_source_folder": source_out.name}
    source_out.mkdir(parents=True, exist_ok=True)
    old = read_checkpoint(cp_path) or {}
    # Keep the last committed marker, but do not nest an unbounded chain of interrupted markers.
    previous = old.get("previous_checkpoint", {}) if old.get("_resume_state") == "IN_PROGRESS" else old
    write_checkpoint(cp_path, {"study_id": study_id, "pipeline_version": PIPELINE_VERSION,
                              "extractor_revision": EXTRACTOR_REVISION, "source_signature": sig,
                              "_resume_state": "IN_PROGRESS", "previous_checkpoint": previous})
    key = hashlib.sha256(json.dumps([str(path), sig, patch_fingerprint()], sort_keys=True).encode()).hexdigest()[:20]
    work = LOCAL_WORK_DIR / key
    work.mkdir(parents=True, exist_ok=True)
    local_base = work / "base"
    marker = read_json_if_exists(local_base / "BASE_COMPLETE.json")
    meta = source_metadata(study_id, row, sig, ftype)
    core_bad = bool(reusable and (reusable[2].get("needs_core_reparse") or
                    (reusable[2].get("pdf_pages") and reusable[2].get("parsed_pages") != reusable[2].get("pdf_pages")) or
                    any(v in str(reusable[2].get("conversion_status", "")).lower() for v in ("partial", "failure"))))
    local_qa = (marker or {}).get("qa", {})
    local_bad = bool(local_qa.get("needs_core_reparse") or
                     (local_qa.get("pdf_pages") and local_qa.get("parsed_pages") != local_qa.get("pdf_pages")) or
                     any(v in str(local_qa.get("conversion_status", "")).lower() for v in ("partial", "failure")))
    if marker and marker.get("core_settings") and marker["core_settings"] != core_settings_fingerprint():
        local_bad = True
    use_local = (not forced and _cache_signature_matches(marker, sig) and base_artifacts_usable(local_base, local_qa)
                 and not (local_bad and EXECUTION_MODE == "BATCH"))
    if use_local:
        base, base_subdir, qa = local_base, "parsed_v4_3", marker.get("qa", {})
        action = "RESUME_LOCAL_EXPORT"
    elif reusable and ftype in {"pdf", "docx", "doc_legacy"} and not (core_bad and EXECUTION_MODE == "BATCH"):
        base, base_subdir, qa = reusable
        action = "UPGRADE_FROM_CACHE"
    else:
        if local_base.exists():
            shutil.rmtree(local_base)
        local_base.mkdir()
        # Short local paths isolate model/export work from the Drive mount and long names.
        local_source = work / ("input" + path.suffix.lower())
        shutil.copy2(path, local_source)
        if sig.get("sha256") and sha256_file(local_source) != sig["sha256"]:
            raise IOError("Source changed or was not fully downloaded during local staging")
        if ftype in {"xlsx", "xls", "csv"}:
            qa = parse_tabular_source(local_source, local_base, ftype, {**meta, "encoding_lookup_name": path.name})
            action = "REEXPORT_NATIVE"
        elif EXECUTION_MODE == "BATCH" and ftype == "pdf":
            qa = parse_pdf_source(local_source, local_base, meta)
            action = "PARSE_NEW_OR_RETRY"
        elif EXECUTION_MODE == "BATCH" and ftype == "docx":
            qa = parse_docx_source(local_source, local_base, meta)
            action = "PARSE_NEW_OR_RETRY"
        elif EXECUTION_MODE == "BATCH" and ftype == "doc_legacy":
            converted = try_convert_legacy_doc(local_source, work / "converted")
            if converted is None:
                qa = {**status_base, "source_qa_status": "NEEDS_REVIEW", "issue": "LEGACY_DOC_NEEDS_CONVERSION"}
            else:
                qa = parse_docx_source(converted, local_base, {**meta, "converted_from": path.name})
                qa["warnings"] = list(qa.get("warnings") or []) + ["LEGACY_DOC_CONVERTED_REVIEW_LAYOUT"]
            action = "PARSE_NEW_OR_RETRY"
        else:
            qa = {**status_base, "source_qa_status": "NEEDS_REVIEW", "issue": f"UNSUPPORTED_FILE_TYPE:{ftype}"}
            action = "NEEDS_REVIEW"
        base, base_subdir = local_base, "parsed_v4_3"
        write_json_atomic(local_base / "source_metadata.json", meta)
        write_json_atomic(local_base / "source_qa.json", qa)
        write_json_atomic(local_base / "BASE_COMPLETE.json", {"source_signature": sig, "qa": qa, "extractor_revision": EXTRACTOR_REVISION, "core_settings": core_settings_fingerprint()})
    overlay = work / "quality"
    if overlay.exists():
        shutil.rmtree(overlay)
    qa = enrich_saved_outputs(path, base, overlay, meta, qa, base_subdir)
    if not source_signatures_match(sig, source_signature_from_row(row)):
        raise IOError("Source changed during processing; completed outputs were not committed")
    if base == local_base:
        publish_directory(local_base, source_out / base_subdir)
    publish_directory(overlay, source_out / QUALITY_DIR_NAME)
    payload = {"study_id": study_id, "pipeline_version": PIPELINE_VERSION, "extractor_revision": EXTRACTOR_REVISION,
               "patch_fingerprint": patch_fingerprint(), "source_signature": sig, "qa": qa,
               "base_parsed_subdir": base_subdir, "quality_subdir": QUALITY_DIR_NAME,
               "output_files": output_file_manifest(source_out, base_subdir),
               "completed_at_utc": datetime.now(timezone.utc).isoformat(), "_resume_state": "COMPLETE"}
    write_checkpoint(cp_path, payload)
    # Only committed/published local work can be removed. Failed work remains recoverable.
    shutil.rmtree(work, ignore_errors=True)
    print(" ", action, path.name, "->", qa.get("source_qa_status"))
    return {**qa, "_action": action, "_source_folder": source_out.name}


In [ ]:
# 16. Read-only database contract, honest QA, and evidence-linked semantic handoff.
# Exact column names inspected in TFM_ML_DATABASE_V3 on 2026-09-23.
DB_FIELD_SCHEMA = {
    "experiments": "experiment_id|study_id|experiment_label|crop|cultivar|growth_stage|system|substrate|season|country|design_notes|updated_at",
    "experiment_context": "experiment_id|study_id|site_id|season_year|season_label|latitude|longitude|environment_type|growth_facility|soil_or_soilless|soil_texture|substrate_composition|pot_volume_l|plant_density|spacing_raw|photoperiod_h|ppfd_umol_m2_s|light_source|relative_humidity_pct|vpd_kpa|co2_ppm|soil_ph|soil_ec_ds_m|organic_matter_pct|irrigation_method|nutrient_solution|fertilization_raw|randomization_unit|experimental_unit|replicate_structure|context_notes|updated_at",
    "arms": "arm_id|experiment_id|arm_label|arm_role|pairing_key|stress_present|stress_type_raw|stress_type|stress_level_raw|stress_value_numeric|stress_unit|stress_duration|stress_duration_unit|irrigation_fraction|field_capacity_pct|soil_water_content_pct|water_potential_mpa|nacl_mm|irrigation_ec_ds_m|substrate_ec_ds_m|day_temperature_c|night_temperature_c|control_temperature_c|temperature_delta_c|biostimulant_present|biostimulant_class_raw|biostimulant_class|biostimulant_source|product|dose_raw|dose_numeric|dose_unit|route|frequency|application_count|notes|updated_at",
    "arm_components": "component_id|arm_id|experiment_id|component_order|component_role|component_name_raw|component_name_standard|component_class|component_source|organism_species|strain|formulation|composition_raw|dose_raw|dose_numeric|dose_unit|concentration_numeric|concentration_unit|application_route|application_stage|application_timing_raw|frequency_raw|application_count|component_notes|verification_status|updated_at",
    "measurements": "measurement_id|natural_key|experiment_id|arm_id|outcome_raw_name|outcome_standard_name|outcome_family|organ|timepoint_raw|timepoint_numeric|timepoint_unit_raw|timepoint_unit|mean|sd|se|n|unit_raw|unit|stat_basis|provenance_id|verification_status|qa_status|notes|updated_at",
    "raw_observations": "observation_id|study_id|experiment_id|arm_id|measurement_id|sample_id|biological_replicate_id|technical_replicate_id|outcome_raw_name|outcome_standard_name|outcome_family|organ|timepoint_raw|value|unit_raw|unit|data_level|aggregation_status|source_file_id|provenance_id|verification_status|qa_status|notes|updated_at",
    "provenance_qa": "provenance_id|entity_type|entity_id|source_file_id|source_type|source_page|source_table|source_figure|source_sheet|source_cell_range|source_quote|extraction_method|extractor|confidence|verification_status|verified_by|verified_at|qa_notes|updated_at",
    "data_assets": "asset_id|study_id|source_file_id|asset_role|asset_type|assay_domain|assay_platform|data_granularity|row_count_or_features|sample_count|sample_mapping_available|raw_or_processed|normalization_reported|primary_ml_eligible|mechanistic_only|linked_experiment_id|linked_arm_mapping|provenance_notes|verification_status|updated_at",
    "risk_of_bias": "rob_id|study_id|experiment_id|domain|judgement|supporting_text|source_file_id|source_page|provenance_id|assessor|verification_status|overall_study_flag|overall_experiment_flag|notes|assessed_at|updated_at",
    "study_relations": "relation_id|study_id_a|study_id_b|relation_type|shared_experiment_id|shared_dataset|shared_site|shared_season|evidence|source_file_id|verification_status|notes|updated_at|created_at",
    "contrasts": "contrast_id|study_id|experiment_id|outcome_standard_name|outcome_family|organ|timepoint_key|unit|contrast_type|measurement_c_id|measurement_s_id|measurement_b_id|measurement_sb_id|arm_c_id|arm_s_id|arm_b_id|arm_sb_id|mean_c|mean_s|mean_b|mean_sb|effect_metric|effect_value|variance|se|variance_source|dependency_cluster_id|input_verification_status|computable|status|notes|generated_at",
    "contrast_terms": "contrast_term_id|contrast_id|measurement_id|arm_id|coefficient|term_role|dependency_cluster_id|shared_control_id|notes|verification_status|updated_at|generated_at",
    "outcome_dictionary": "outcome_raw_name|outcome_standard_name|outcome_family|construct_definition|organ_scope|preferred_unit|directionality_rule|primary_endpoint_priority|aggregation_rule|transform_rule|higher_is_better_applicable|eligible_for_primary_ml|eligible_for_secondary_ml|mechanistic_only|synonyms|mapping_status|notes|updated_at",
}
DB_FIELD_SCHEMA = {key: value.split("|") for key,value in DB_FIELD_SCHEMA.items()}
PROJECT_CONTRACT = {
    "schema_version": "3.4.0", "corpus_version": "v1.5", "inspected_at": "2026-09-23",
    "source_workbook": "https://docs.google.com/spreadsheets/d/1cD_iZVLF4hMeeAP7_DkaoT7YtjMvThn2k1WzwgwvR_g/edit",
    "columns": DB_FIELD_SCHEMA,
    "master_write_enabled": False,
    "provenance_rule": "Use the active current corpus file ID, not a historical archive ID, plus page/table/figure/sheet/cell/quote locator.",
    "primary_target": "Matched tomato C/S/B/SB: (SB-S)-(B-C). SB_vs_S is secondary; study class does not establish design eligibility.",
    "stress_rule": "ABIOTIC and BIOTIC; retain exact pathogen/disease wording, mixtures and multi-stressor designs. Fusarium is a prespecified subgroup, not an automatic relabelling rule.",
    "temporal_rule": "Retain frozen dataset_role and temporal_eligibility. Unknown year remains provisional; reviewed post-2023 studies are not truly unseen EXTERNAL_TEST.",
    "raw_rule": "Cell-level values are not automatically biological replicates. Preserve biological/technical replicate identity and avoid pseudoreplication.",
    "ml_rule": "Current corpus is TRAIN_DEV. Grouped CV; no internal TEST split. Keep post-treatment mechanistic variables out of ordinary pre-treatment predictors.",
    "schema_extensions": "Information not represented by an existing column stays in evidence/extension JSON. No master columns are silently created.",
}


def validate_semantic_bundle(bundle, active_source_ids):
    """Structural preflight only; this does not verify measurements or scientific interpretation."""
    if not isinstance(bundle, dict):
        return ["EXPECTED_BUNDLE_OBJECT"]
    issues, ids, clean = [], {}, {}
    primary = {"experiments":"experiment_id", "arms":"arm_id", "measurements":"measurement_id", "provenance_qa":"provenance_id",
               "arm_components":"component_id", "raw_observations":"observation_id", "data_assets":"asset_id",
               "risk_of_bias":"rob_id", "study_relations":"relation_id", "contrasts":"contrast_id", "contrast_terms":"contrast_term_id"}
    for table, records in bundle.items():
        if table not in DB_FIELD_SCHEMA:
            if table != "extensions":
                issues.append(f"UNKNOWN_TABLE:{table}")
            continue
        if not isinstance(records, list) or any(not isinstance(r, dict) for r in records):
            issues.append(f"EXPECTED_RECORD_LIST:{table}")
            continue
        clean[table] = records
        key, seen = primary.get(table), set()
        for row in records:
            extra = set(row)-set(DB_FIELD_SCHEMA[table])
            if extra:
                issues.append(f"UNKNOWN_COLUMNS:{table}:{sorted(map(str,extra))}")
            if key:
                value = row.get(key)
                if not isinstance(value, str) or not value.strip():
                    issues.append(f"MISSING_OR_INVALID_ID:{table}")
                elif value in seen:
                    issues.append(f"DUPLICATE_ID:{table}:{value}")
                else:
                    seen.add(value)
            value = row.get("source_file_id")
            if value and (not isinstance(value, str) or value not in active_source_ids):
                issues.append(f"NONACTIVE_SOURCE_ID:{table}:{value}")
        ids[table] = seen
    for table, field, parent in [("arms","experiment_id","experiments"), ("measurements","arm_id","arms"),
                                  ("measurements","experiment_id","experiments"), ("measurements","provenance_id","provenance_qa"),
                                  ("arm_components","arm_id","arms"), ("contrast_terms","contrast_id","contrasts")]:
        for row in clean.get(table, []):
            value = row.get(field)
            if not isinstance(value, str) or value not in ids.get(parent, set()):
                issues.append(f"BROKEN_LINK:{table}.{field}:{value}")
    for row in clean.get("provenance_qa", []):
        if not row.get("source_file_id") or not any(row.get(f) for f in ["source_page","source_table","source_figure","source_sheet","source_cell_range","source_quote"]):
            issues.append(f"MISSING_SOURCE_LOCATOR:{row.get('provenance_id')}")
    natural = []
    for row in clean.get("measurements", []):
        key = row.get("natural_key")
        if key is not None and not isinstance(key, str):
            issues.append("INVALID_MEASUREMENT_NATURAL_KEY")
        elif key:
            natural.append(key)
    if len(natural) != len(set(natural)):
        issues.append("DUPLICATE_MEASUREMENT_NATURAL_KEY")
    return issues


def aggregate_study_qa(study_id, source_qas, source_rows):
    n = len(source_rows)
    full = int(source_rows["role"].eq("FULLTEXT").sum()) if n else 0
    supplements = int(source_rows["role"].eq("SUPPLEMENT").sum()) if n else 0
    fail = sum(q.get("source_qa_status") == "FAIL" for q in source_qas)
    review = sum(q.get("source_qa_status") == "NEEDS_REVIEW" for q in source_qas)
    usable = sum(q.get("parser") in {"docling_pdf","docling_docx","native_tabular"} and q.get("source_qa_status") in {"PASS","NEEDS_REVIEW"} for q in source_qas)
    summed = {k:sum(int(q.get(k) or 0) for q in source_qas) for k in ["tables_detected","tables_exported","table_images_exported",
              "figures_detected","figures_exported","quantitative_figures_detected","quantitative_figures_exported","semantic_text_candidates"]}
    def coverage(done, expected):
        return summed[done] / summed[expected] if summed[expected] else 1.0
    table_cov = coverage("tables_exported","tables_detected")
    table_img_cov = coverage("table_images_exported","tables_detected")
    fig_cov = coverage("figures_exported","figures_detected")
    qfig_cov = coverage("quantitative_figures_exported","quantitative_figures_detected")
    complete = bool(n and full and not fail and usable == n and min(table_cov,table_img_cov,fig_cov,qfig_cov) == 1.0)
    if EXECUTION_MODE == "AUDIT_EXISTING":
        status = "AUDIT_ONLY_NOT_PROCESSED"
    elif not n:
        status = "FAIL_NO_SOURCE_FILES"
    elif not full:
        status = "FAIL_NO_FULLTEXT"
    elif fail:
        status = "FAIL"
    elif review:
        status = "NEEDS_REVIEW"
    elif not complete:
        status = "INCOMPLETE"
    else:
        status = "SOURCE_LAYER_COMPLETE__FIGURE_RECONCILIATION_PENDING" if summed["quantitative_figures_detected"] else "SOURCE_LAYER_COMPLETE"
    return {"study_id":study_id, "pipeline_version":PIPELINE_VERSION, "extractor_revision":EXTRACTOR_REVISION,
            "database_spec":DATABASE_SPEC, "expected_source_files":n, "processed_source_files":usable,
            "attempted_source_files":len(source_qas), "fulltext_files_expected":full,"supplement_files_expected":supplements,
            "source_file_coverage":usable/n if n else 0.0, "attempted_source_file_coverage":len(source_qas)/n if n else 0.0,
            **summed, "table_asset_coverage":table_cov, "table_image_coverage":table_img_cov,
            "figure_asset_coverage":fig_cov,"quantitative_figure_asset_coverage":qfig_cov,
            "failed_sources":fail,"needs_review_sources":review,"source_layer_complete":complete,
            "study_source_qa_status":status, "semantic_qa_status":"PENDING_VERIFICATION",
            "measurement_recall":None,"numeric_accuracy":None,"provenance_completeness":None,
            "completed_at_utc":datetime.now(timezone.utc).isoformat()}


def write_study_handoff(study_out, study_id, study_qa, source_rows, source_qas):
    sources, review = [], []
    for row, current_qa in zip(source_rows.to_dict("records"), source_qas):
        source_out = source_directory(row, study_out)
        cp = read_checkpoint(source_out / "SOURCE_COMPLETE.json") or {}
        valid_generation = cp.get("_resume_state") == "COMPLETE" and cp.get("extractor_revision") == EXTRACTOR_REVISION and current_qa.get("_action") not in {"SOURCE_ERROR","MISSING_SOURCE","NEEDS_REPARSE"}
        valid_generation = bool(valid_generation and cp.get("patch_fingerprint") == patch_fingerprint() and
                                source_signatures_match(cp.get("source_signature", {}), source_signature_from_row(row)))
        quality = source_out / QUALITY_DIR_NAME
        inv = (read_json_if_exists(quality / "source_inventory.json") or {}) if valid_generation else {}
        base_subdir = cp.get("base_parsed_subdir", "parsed")
        paths = {}
        if valid_generation:
            for name in ("source_inventory.json","source_qa.json","source_metadata.json","text_items.jsonl","semantic_text_candidates.csv","text_chunks.jsonl","evidence_index.jsonl","table_inventory.csv","figure_inventory.csv","figure_digitization_queue.csv"):
                p = quality / name
                if p.is_file():
                    paths[name] = p.relative_to(study_out).as_posix()
            for name in ("docling_document.json","document.md","document.txt","sheet_inventory.csv"):
                p = source_out / base_subdir / name
                if p.is_file():
                    paths[name] = p.relative_to(study_out).as_posix()
        sources.append({"source_file_id":row.get("source_file_id"),"source_folder":source_out.name,
                        "current_generation_available":valid_generation,"source_metadata":inv.get("source",{}),
                        "source_qa":current_qa,"paths":paths})
        if current_qa.get("source_qa_status") != "PASS":
            review.append({"review_type":"SOURCE_QA","source_file_id":row.get("source_file_id"),
                           "reason":current_qa.get("issue") or current_qa.get("error") or current_qa.get("warnings")})
        for table in inv.get("tables",[]):
            review.append({"review_type":"TABLE_SEMANTIC_RECONCILIATION","source_file_id":row.get("source_file_id"),"table_index":table.get("table_index"),"reason":"Map outcomes, units, treatment arms, timepoints, SD/SE and n; preserve source cell spans"})
        for figure in inv.get("figures",[]):
            if figure.get("likely_quantitative"):
                review.append({"review_type":"FIGURE_DIGITIZATION_OR_DUPLICATE_CHECK","source_file_id":row.get("source_file_id"),"figure_index":figure.get("figure_index"),"reason":"Candidate only; not a verified database measurement"})
    context = [{k:v for k,v in r.items() if k not in {"source_path","size_bytes","modified_time_ns"}} for r in source_rows.to_dict("records")]
    handoff = {"handoff_version":EXTRACTOR_REVISION,"study_id":study_id,"database_spec":DATABASE_SPEC,
               "project_contract":PROJECT_CONTRACT,"screening_manifest":context,"study_source_qa":study_qa,
               "source_priority":SOURCE_PRIORITY,"extraction_targets":EXTRACTION_TARGETS,"sources":sources,
               "semantic_completion_rule":"Each numerical text/table/figure asset must be extracted, reconciled as duplicate, explicitly excluded with a source-supported reason, or queued for review.",
               "required_semantic_output_schema":"DATABASE_SCHEMA_V3_4_0.v4_3.json",
               "unsupported_schema_fields":"Keep in evidence/extension JSON; do not discard pathogen detail, timepoint anchors, uncertainty definitions or mixtures.",
               "master_write_enabled":False}
    write_json_atomic(study_out / "STUDY_HANDOFF.v4_3.json",handoff)
    _atomic_write_text(study_out / "HUMAN_REVIEW_QUEUE.v4_3.csv",pd.DataFrame(review).to_csv(index=False))
    template = study_out / "STRUCTURED_EXTRACTION_TEMPLATE.v4_3.json"
    if not template.exists():
        write_json_atomic(template,{**{name:[] for name in DB_FIELD_SCHEMA},"extensions":{}})
    write_json_atomic(RUN_LOGS_DIR / "DATABASE_SCHEMA_V3_4_0.v4_3.json",PROJECT_CONTRACT)


In [ ]:
# 16B. Repair/enrich the saved evidence layer without invoking Docling conversion.
import base64
from io import BytesIO


def write_json_atomic(path, obj):
    _atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=2, default=str))


def raw_bbox_rect(prov, pages):
    """Docling source coordinates -> normalized top-left PDF coordinates."""
    bbox = prov.get("bbox") or {}
    l, t, r, b = (float(bbox[k]) for k in ("l", "t", "r", "b"))
    if "BOTTOM" in str(bbox.get("coord_origin", "")).upper():
        page = pages.get(str(prov.get("page_no")), pages.get(prov.get("page_no"), {}))
        h = float(page["size"]["height"])
        t, b = h-t, h-b
    return fitz.Rect(min(l,r), min(t,b), max(l,r), max(t,b))


def image_from_ref(ref, base_dir):
    if not isinstance(ref, dict) or not ref.get("uri"):
        return None
    uri = str(ref["uri"])
    if uri.startswith("data:image/") and ";base64," in uri:
        with Image.open(BytesIO(base64.b64decode(uri.split(",", 1)[1], validate=True))) as img:
            return img.copy()
    # Never fetch a URL contained in a source document. Only local relative artifacts.
    if "://" not in uri and not Path(uri).is_absolute():
        target = (base_dir / uri).resolve()
        if target.is_relative_to(base_dir.resolve()) and target.is_file():
            with Image.open(target) as img:
                return img.copy()
    return None


def valid_image(path):
    if not path.is_file():
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except (OSError, ValueError, SyntaxError):
        return False


def raw_caption(item, raw_doc):
    parts = []
    for caption in item.get("captions", []):
        ref = caption.get("$ref", "") if isinstance(caption, dict) else ""
        try:
            group, index = ref.strip("#/").split("/")
            text = raw_doc[group][int(index)].get("text", "")
            if text:
                parts.append(text)
        except (KeyError, IndexError, ValueError, TypeError):
            continue
    return " ".join(parts)


def restore_item_image(item, raw_doc, base_dir, source_path, destination):
    img = image_from_ref(item.get("image"), base_dir)
    if img is None and item.get("prov"):
        prov = item["prov"][0]
        pages = raw_doc.get("pages", {})
        page = pages.get(str(prov["page_no"]), {})
        page_img = image_from_ref(page.get("image"), base_dir)
        if page_img is not None:
            rect = raw_bbox_rect(prov, pages)
            size = page["size"]
            sx, sy = page_img.width / float(size["width"]), page_img.height / float(size["height"])
            box = (max(0, int(rect.x0*sx)), max(0, int(rect.y0*sy)),
                   min(page_img.width, math.ceil(rect.x1*sx)), min(page_img.height, math.ceil(rect.y1*sy)))
            if box[2] > box[0] and box[3] > box[1]:
                img = page_img.crop(box)
            page_img.close()
        elif source_path and source_path.suffix.lower() == ".pdf" and source_path.exists():
            rect = raw_bbox_rect(prov, pages)
            if render_pdf_rect(source_path, int(prov["page_no"]), rect, destination):
                return "OK_RENDERED_FROM_PAGE"
    if img is not None:
        try:
            img.save(destination)
        finally:
            img.close()
        return "OK_RECOVERED_FROM_CACHE"
    return "MISSING"


def raw_text_inventory(raw_doc):
    """Walk saved reading-order references; retain also unlinked/furniture text."""
    ordered, seen = [], set()
    def visit(node):
        if not isinstance(node, dict):
            return
        ref = node.get("$ref")
        if ref:
            if ref in seen:
                return
            seen.add(ref)
            parts = ref.strip("#/").split("/")
            try:
                target = raw_doc
                for part in parts:
                    target = target[int(part)] if isinstance(target, list) else target[part]
            except (KeyError, IndexError, ValueError, TypeError):
                return
            if parts[0] == "texts":
                ordered.append(target)
            visit(target)
        else:
            for child in node.get("children", []):
                visit(child)
    visit(raw_doc.get("body", {}))
    visit(raw_doc.get("furniture", {}))
    present = {x.get("self_ref") for x in ordered}
    ordered.extend(x for x in raw_doc.get("texts", []) if x.get("self_ref") not in present)
    result, section = [], ""
    for idx, item in enumerate(ordered, 1):
        text = str(item.get("text") or "")
        if not text.strip():
            continue
        label = str(item.get("label", ""))
        if label in {"section_header", "title"}:
            section = text[:300]
        prov = item.get("prov", [])
        p = prov[0] if prov else {}
        bbox = p.get("bbox", {})
        result.append({"text_item_id": idx, "text_ref": item.get("self_ref"), "label": label,
                       "section": section, "text": text, "page_no": p.get("page_no"),
                       "bbox_l": bbox.get("l"), "bbox_t": bbox.get("t"),
                       "bbox_r": bbox.get("r"), "bbox_b": bbox.get("b"),
                       "bbox_origin": bbox.get("coord_origin"), "all_provenance": prov,
                       "content_layer": item.get("content_layer"), **candidate_flags(text)})
    return result


def raw_table_grid(table):
    data = table.get("data") or {}
    cells = data.get("table_cells", [])
    nr, nc = int(data.get("num_rows", 0)), int(data.get("num_cols", 0))
    if nr < 0 or nc < 0 or nr*nc > 10_000_000:
        raise ValueError("Invalid or excessively large detected document table dimensions")
    grid = [[""] * nc for _ in range(nr)]
    for cell in cells:
        r, c = int(cell["start_row_offset_idx"]), int(cell["start_col_offset_idx"])
        re_, ce = int(cell["end_row_offset_idx"]), int(cell["end_col_offset_idx"])
        if not (0 <= r < re_ <= nr and 0 <= c < ce <= nc):
            raise ValueError("Detected table cell span is outside its grid")
        # Place merged text at the anchor only. Original spans remain in cells.jsonl.
        grid[r][c] = str(cell.get("text", ""))
    return grid, cells


def evidence_id(source_meta, locator):
    identity = [source_meta.get("source_file_id"), source_meta.get("sha256"), locator]
    return "ev_" + hashlib.sha256(json.dumps(identity, sort_keys=True, default=str).encode()).hexdigest()[:24]


def enrich_saved_outputs(source_path, base_dir, out_dir, source_meta, original_qa, base_subdir="parsed"):
    """Write a separate corrected overlay. The base parsed directory is not edited."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for sub in ("tables", "figures"):
        (out_dir / sub).mkdir(exist_ok=True)
    inv = read_json_if_exists(base_dir / "source_inventory.json") or {}
    qa = dict(original_qa or inv.get("qa") or {})
    warnings = list(qa.get("warnings") or [])
    raw_doc = read_json_if_exists(base_dir / "docling_document.json")
    is_doc = qa.get("parser") in {"docling_pdf", "docling_docx"}
    if is_doc and (not isinstance(raw_doc, dict) or not isinstance(raw_doc.get("texts"), list)):
        raise ValueError("Missing/invalid cached Docling document; source needs selective reparse")
    raw_doc = raw_doc or {}
    evidence, files = [], []
    def overlay_rel(name):
        return (Path(QUALITY_DIR_NAME) / name).as_posix()
    def base_rel(name):
        return (Path(base_subdir) / name).as_posix()
    def record_evidence(kind, locator, **extra):
        evidence.append({"evidence_id": evidence_id(source_meta, locator), "study_id": source_meta.get("study_id"),
                         "source_file_id": source_meta.get("source_file_id"), "source_sha256": source_meta.get("sha256"),
                         "kind": kind, "locator": locator, "verification_status": "UNVERIFIED",
                         "semantic_status": "PENDING", **extra})
    text_rows = raw_text_inventory(raw_doc) if is_doc else []
    if is_doc:
        pd.DataFrame(text_rows).to_csv(out_dir / "text_items.csv", index=False)
        candidates = [r for r in text_rows if any(v for k,v in r.items() if k.startswith("is_") and isinstance(v, bool))]
        pd.DataFrame(candidates).to_csv(out_dir / "semantic_text_candidates.csv", index=False)
        write_text_chunks(text_rows, out_dir / "text_chunks.jsonl")
        with (out_dir / "text_items.jsonl").open("w", encoding="utf-8") as h:
            for row in text_rows:
                h.write(json.dumps(row, ensure_ascii=False) + "\n")
        qa.update(text_items=len(text_rows), semantic_text_candidates=len(candidates))
        for row in candidates:
            record_evidence("TEXT_CANDIDATE", {"text_ref": row["text_ref"], "provenance": row["all_provenance"]},
                            source_quote=row["text"], section=row["section"],
                            candidate_flags={k:v for k,v in row.items() if k.startswith("is_")})
        if not text_rows:
            warnings.append("NO_TEXT_EXTRACTED")

    table_rows = []
    old_tables = {int(x.get("table_index", i)): x for i,x in enumerate(inv.get("tables", []), 1)}
    for idx, table in enumerate(raw_doc.get("tables", []), 1):
        info = dict(old_tables.get(idx, {}))
        info.update(table_index=idx, all_provenance=table.get("prov", []), table_ref=table.get("self_ref"))
        if not info.get("caption"):
            info["caption"] = raw_caption(table, raw_doc)
        if table.get("prov"):
            info["page_no"] = table["prov"][0].get("page_no")
        name = f"table_{idx:03d}"
        grid, raw_cells = raw_table_grid(table)
        raw_csv = out_dir / "tables" / (name + "__raw_grid.csv")
        with raw_csv.open("w", encoding="utf-8", newline="") as h:
            csv.writer(h).writerows(grid)
        cell_path = out_dir / "tables" / (name + "__cells.jsonl")
        with cell_path.open("w", encoding="utf-8") as h:
            for cell_idx, cell in enumerate(raw_cells, 1):
                item = {"source_file_id": source_meta.get("source_file_id"), "table_index": idx,
                        "cell_index": cell_idx, "table_provenance": table.get("prov", []), **cell}
                h.write(json.dumps(item, ensure_ascii=False) + "\n")
        info.update(raw_grid_relpath=overlay_rel(Path("tables") / raw_csv.name),
                    cells_relpath=overlay_rel(Path("tables") / cell_path.name),
                    raw_cell_count=len(raw_cells), raw_grid_has_header=False)
        original_csv = base_dir / "tables" / str(info.get("csv_file") or "")
        if info.get("export_status") == "OK" and original_csv.is_file() and original_csv.stat().st_size:
            info["csv_relpath"] = base_rel(Path("tables") / original_csv.name)
        else:
            info["previous_export_error"] = info.get("error")
            info.update(csv_file=raw_csv.name, csv_relpath=overlay_rel(Path("tables") / raw_csv.name),
                        csv_representation="RAW_DETECTED_GRID_WITH_SPANS_IN_CELLS_JSONL",
                        export_status="OK" if raw_cells else "FAILED",
                        recovered_without_reconversion=bool(raw_cells))
            info.pop("error", None)
        info["numeric_cell_count"] = sum(bool(NUMERIC_TOKEN_RE.fullmatch(str(c.get("text", "")).strip())) for c in raw_cells)
        image_name = str(info.get("table_image_file") or name + ".png")
        old_image = base_dir / "tables" / image_name
        if valid_image(old_image):
            info.update(table_image_status="OK", table_image_file=image_name,
                        image_relpath=base_rel(Path("tables") / image_name))
        else:
            destination = out_dir / "tables" / (name + ".png")
            try:
                status = restore_item_image(table, raw_doc, base_dir, source_path, destination)
            except Exception as e:
                status = "FAILED"
                info["table_image_error"] = repr(e)
            info["table_image_status"] = status
            if status.startswith("OK"):
                info.update(table_image_file=destination.name, image_relpath=overlay_rel(Path("tables") / destination.name))
            else:
                warnings.append(f"TABLE_AUDIT_IMAGE_MISSING:{idx}")
        table_rows.append(info)
        record_evidence("TABLE", {"table_ref": table.get("self_ref"), "provenance": table.get("prov", [])},
                        caption=info.get("caption"), csv_path=info["csv_relpath"], cells_path=info["cells_relpath"],
                        image_path=info.get("image_relpath"))

    fig_rows, figure_queue = [], []
    old_figures = {int(x.get("figure_index", i)): x for i,x in enumerate(inv.get("figures", []), 1)}
    for idx, picture in enumerate(raw_doc.get("pictures", []), 1):
        info = dict(old_figures.get(idx, {}))
        provs = picture.get("prov", [])
        info.update(figure_index=idx, figure_ref=picture.get("self_ref"), all_provenance=provs)
        if not info.get("caption"):
            info["caption"] = raw_caption(picture, raw_doc)
        image_name = str(info.get("image_file") or f"figure_{idx:03d}.png")
        old_image = base_dir / "figures" / image_name
        if valid_image(old_image):
            info.update(image_status="OK", image_file=image_name, image_relpath=base_rel(Path("figures") / image_name))
        else:
            destination = out_dir / "figures" / f"figure_{idx:03d}.png"
            try:
                status = restore_item_image(picture, raw_doc, base_dir, source_path, destination)
            except Exception as e:
                status = "FAILED"
                info["image_error"] = repr(e)
            info["image_status"] = status
            if status.startswith("OK"):
                info.update(image_file=destination.name, image_relpath=overlay_rel(Path("figures") / destination.name))
        rect = None
        if provs:
            try:
                page_no = int(provs[0]["page_no"])
                rect = raw_bbox_rect(provs[0], raw_doc.get("pages", {}))
                h = float(raw_doc["pages"][str(page_no)]["size"]["height"])
                info.update(page_no=page_no, nearby_page_text=nearby_text_for_bbox(text_rows, page_no, rect, page_height=h))
            except (KeyError, TypeError, ValueError) as e:
                info["nearby_text_warning"] = repr(e)
        # Reuse native figure evidence; recover missing export channels without OCR/model conversion.
        words_name = f"figure_{idx:03d}__native_text.csv"
        vectors_name = f"figure_{idx:03d}__vector_primitives.json"
        for field, filename in (("native_text_relpath", words_name), ("vector_primitives_relpath", vectors_name)):
            if (base_dir / "figures" / filename).is_file():
                info[field] = base_rel(Path("figures") / filename)
        if qa.get("parser") == "docling_pdf" and rect is not None and ("native_text_relpath" not in info or "vector_primitives_relpath" not in info):
            if source_path and source_path.is_file():
                try:
                    words = fitz_words_in_rect(source_path, int(provs[0]["page_no"]), rect)
                    drawings = vector_primitives_in_rect(source_path, int(provs[0]["page_no"]), rect)
                    if "native_text_relpath" not in info:
                        pd.DataFrame(words, columns=["text","x0","y0","x1","y1"]).to_csv(out_dir / "figures" / words_name, index=False)
                        info["native_text_relpath"] = overlay_rel(Path("figures") / words_name)
                    if "vector_primitives_relpath" not in info:
                        write_json_atomic(out_dir / "figures" / vectors_name, drawings)
                        info["vector_primitives_relpath"] = overlay_rel(Path("figures") / vectors_name)
                    info["native_text_token_count"] = len(words)
                    info["vector_drawing_count"] = len(drawings)
                except Exception as e:
                    warnings.append(f"FIGURE_NATIVE_CHANNEL_EXPORT_FAILED:{idx}")
                    info["native_channel_error"] = repr(e)
            else:
                warnings.append(f"FIGURE_NATIVE_CHANNELS_NOT_CHECKED:{idx}")
        info["likely_quantitative"] = bool(info.get("likely_quantitative") or QUANT_CAPTION_RE.search(info.get("caption") or ""))
        if not old_figures.get(idx) and info.get("image_relpath"):
            recovered_image = out_dir / "figures" / image_name
            if not recovered_image.is_file():
                recovered_image = old_image
            try:
                with Image.open(recovered_image) as img:
                    geometry = image_geometry_features(img)
                info.update(geometry)
                info["likely_quantitative"] |= bool(geometry.get("chart_geometry_hint"))
            except (OSError, ValueError):
                pass
        auto_name = f"figure_{idx:03d}__auto_vector_bars.csv"
        if (base_dir / "figures" / auto_name).is_file():
            info["auto_vector_candidates_relpath"] = base_rel(Path("figures") / auto_name)
        if info["likely_quantitative"]:
            figure_queue.append({"figure_index": idx, "page_no": info.get("page_no"), "caption": info.get("caption"),
                                 "image_file": info.get("image_file"), "image_relpath": info.get("image_relpath"),
                                 "reason": "quantitative_figure_requires_semantic_reconciliation", "verified": False,
                                 "auto_status": info.get("digitization_status", "NOT_DIGITIZED")})
        fig_rows.append(info)
        record_evidence("FIGURE", {"figure_ref": picture.get("self_ref"), "provenance": provs},
                        caption=info.get("caption"), image_path=info.get("image_relpath"),
                        likely_quantitative=info["likely_quantitative"],
                        candidate_values_path=info.get("auto_vector_candidates_relpath"))

    sheets = inv.get("sheets", [])
    for sheet in sheets:
        for field in ("csv_file", "cell_provenance_file", "typed_cell_provenance_file", "record_layout_file"):
            if sheet.get(field):
                sheet[field + "_relpath"] = base_rel(sheet[field])
        record_evidence("SHEET", {"sheet_name": sheet.get("sheet_name")},
                        csv_path=sheet.get("csv_file_relpath"), cells_path=sheet.get("typed_cell_provenance_file_relpath") or sheet.get("cell_provenance_file_relpath"),
                        rows=sheet.get("n_rows"), columns=sheet.get("n_cols"),
                        biological_sample_mapping_status="NOT_INFERRED")
    if is_doc:
        pd.DataFrame(table_rows).to_csv(out_dir / "table_inventory.csv", index=False)
        pd.DataFrame(fig_rows).to_csv(out_dir / "figure_inventory.csv", index=False)
        pd.DataFrame(figure_queue).to_csv(out_dir / "figure_digitization_queue.csv", index=False)
        qa.update(tables_detected=len(table_rows), tables_exported=sum(x.get("export_status") == "OK" for x in table_rows),
                  table_images_exported=sum(str(x.get("table_image_status", "")).startswith("OK") for x in table_rows),
                  figures_detected=len(fig_rows), figures_exported=sum(str(x.get("image_status", "")).startswith("OK") for x in fig_rows),
                  quantitative_figures_detected=sum(x["likely_quantitative"] for x in fig_rows),
                  quantitative_figures_exported=sum(x["likely_quantitative"] and str(x.get("image_status", "")).startswith("OK") for x in fig_rows))
        qn = qa["quantitative_figures_detected"]
        qa["quantitative_figure_asset_coverage"] = qa["quantitative_figures_exported"] / qn if qn else 1.0
        qa["table_image_coverage"] = qa["table_images_exported"] / len(table_rows) if table_rows else 1.0
        hard = qa["tables_exported"] != len(table_rows) or qa["figures_exported"] != len(fig_rows)
        core_incomplete = qa.get("needs_core_reparse", False) or (qa.get("pdf_pages") and qa.get("parsed_pages") != qa.get("pdf_pages")) or any(s in str(qa.get("conversion_status", "")).lower() for s in ("partial", "failure"))
        hard = bool(hard or core_incomplete)
        qa["needs_core_reparse"] = bool(core_incomplete)
        qa["source_qa_status"] = "FAIL" if hard else ("NEEDS_REVIEW" if warnings else "PASS")
        qa["table_export_failures"] = [t["table_index"] for t in table_rows if t.get("export_status") != "OK"]
        if not hard:
            for field in ("error", "traceback"):
                if field in qa:
                    qa["previous_export_" + field] = qa.pop(field)
    qa.update(source_file_id=source_meta.get("source_file_id"), source_file_name=source_meta.get("source_file_name"),
              role=source_meta.get("role"), extractor_revision=EXTRACTOR_REVISION,
              semantic_extraction_status="PENDING", numeric_accuracy=None, measurement_completeness_status="NOT_YET_MEASURED",
              warnings=list(dict.fromkeys(str(w) for w in warnings)))
    write_json_atomic(out_dir / "source_qa.json", qa)
    write_json_atomic(out_dir / "source_metadata.json", source_meta)
    write_json_atomic(out_dir / "source_inventory.json", {"source": source_meta, "qa": qa, "tables": table_rows,
                      "figures": fig_rows, "sheets": sheets, "base_parsed_subdir": base_subdir})
    with (out_dir / "evidence_index.jsonl").open("w", encoding="utf-8") as h:
        for item in evidence:
            h.write(json.dumps(item, ensure_ascii=False, default=str) + "\n")
    return qa

# Checkpoints are valid only for matching content, manifest context, patch, and saved files.
LOCAL_WORK_DIR = Path(os.environ.get("LOCALAPPDATA") or (Path.home() / ".cache")) / "SynerPlant" / "work"


def core_settings_fingerprint():
    options = {"ocr": DO_OCR, "image_scale": IMAGE_SCALE, "table_accurate": TABLE_MODE_ACCURATE,
               "chart_extraction": ENABLE_DOCLING_CHART_EXTRACTION}
    return hashlib.sha256(json.dumps(options, sort_keys=True).encode()).hexdigest()


def patch_fingerprint():
    settings = {"revision": EXTRACTOR_REVISION, "image_scale": IMAGE_SCALE,
                "figure_scale": FIGURE_RENDER_SCALE, "ocr": DO_OCR, "table_accurate": TABLE_MODE_ACCURATE,
                "cell_provenance": EXPORT_CELL_LEVEL_PROVENANCE, "stream_all": STREAM_ALL_CELL_PROVENANCE,
                "cell_limit": MAX_NONEMPTY_CELLS_FOR_LONG_EXPORT,
                "encoding_overrides": DELIMITED_ENCODING_OVERRIDES, "separator_overrides": DELIMITED_SEPARATOR_OVERRIDES,
                "core_settings": core_settings_fingerprint(), "vector_digitization": TRY_VECTOR_BAR_DIGITIZATION, "vector_min_r2": MIN_BAR_DIGITIZATION_R2}
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()


def source_signature_from_row(row):
    path = Path(row["source_path"]) if row.get("source_path") else None
    st = path.stat() if path and path.is_file() else None
    context = {k:row.get(k) for k in ("role", "screening_class", "ml_role", "dataset_role", "temporal_eligibility",
               "first_publication_date", "source_rank", "job_id", "historical_source_file_id")}
    sig = {"source_file_name": clean_str(row.get("source_file_name")), "source_file_id": clean_str(row.get("source_file_id")),
           "size_bytes": int(st.st_size if st else row.get("size_bytes") or 0),
           "modified_time_ns": int(st.st_mtime_ns if st else row.get("modified_time_ns") or 0),
           "manifest_context_hash": hashlib.sha256(json.dumps(context, sort_keys=True, default=str).encode()).hexdigest()}
    if st and HASH_POLICY.upper() != "NONE":
        sig["sha256"] = cached_sha256(path)
    return sig


def output_file_manifest(source_out, base_subdir):
    files = []
    for folder in (source_out / base_subdir, source_out / QUALITY_DIR_NAME):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                files.append({"path": path.relative_to(source_out).as_posix(), "size_bytes": path.stat().st_size})
    return files


def checkpoint_matches(checkpoint_path, current_sig):
    if FORCE_REPROCESS or current_sig.get("source_file_id") in REPARSE_SOURCE_IDS:
        return False
    cp = read_checkpoint(checkpoint_path)
    if not isinstance(cp, dict) or cp.get("_resume_state") != "COMPLETE" or cp.get("pipeline_version") != PIPELINE_VERSION:
        return False
    if cp.get("extractor_revision") != EXTRACTOR_REVISION or cp.get("patch_fingerprint") != patch_fingerprint():
        return False
    if not source_signatures_match(cp.get("source_signature", {}), current_sig):
        return False
    if cp["source_signature"].get("manifest_context_hash") != current_sig.get("manifest_context_hash"):
        return False
    qa = cp.get("qa") or {}
    if qa.get("source_qa_status") not in {"PASS", "NEEDS_REVIEW"} or not cp.get("output_files"):
        return False
    source_out = checkpoint_path.parent
    if not isinstance(cp["output_files"], list):
        return False
    for item in cp["output_files"]:
        if not isinstance(item, dict) or not isinstance(item.get("path"), str) or not isinstance(item.get("size_bytes"), int):
            return False
        relative = Path(item["path"])
        if relative.is_absolute() or ".." in relative.parts:
            return False
        try:
            path = source_out / relative
            if not path.is_file() or path.stat().st_size != item["size_bytes"]:
                return False
        except OSError:
            return False
    return isinstance(read_json_if_exists(source_out / QUALITY_DIR_NAME / "source_qa.json"), dict)


def study_checkpoint_is_current(study_id):
    if FORCE_REPROCESS:
        return False
    cp = read_checkpoint(CHECKPOINT_DIR / f"{study_id}__STUDY_COMPLETE.json") or {}
    if cp.get("extractor_revision") != EXTRACTOR_REVISION or cp.get("_resume_state") != "COMPLETE":
        return False
    if cp.get("patch_fingerprint") != patch_fingerprint():
        return False
    study_out = RUN_EXTRACTED_DIR / safe_name(study_id)
    for name in ("STUDY_QA.v4_3.json", "STUDY_HANDOFF.v4_3.json"):
        if not isinstance(read_json_if_exists(study_out / name), dict):
            return False
    rows = source_rows_for_study(study_id).to_dict("records")
    signatures = [source_signature_from_row(r) for r in rows]
    if not study_signatures_match(cp.get("source_signatures"), signatures):
        return False
    return all(not row.get("manifest_missing_local") and checkpoint_matches(source_directory(row, study_out) / "SOURCE_COMPLETE.json", sig)
               for row,sig in zip(rows,signatures))


def restore_batch_results(path, selected):
    by_id = {}
    for candidate in (path, _resume_local_path(path)):
        try:
            frame = pd.read_csv(candidate, dtype={"study_id":str}, keep_default_na=False)
            for row in frame.to_dict("records"):
                sid = str(row.get("study_id", ""))
                if sid in selected and (sid not in by_id or _resume_timestamp(row) >= _resume_timestamp(by_id[sid])):
                    by_id[sid] = row
        except (OSError, ValueError, pd.errors.ParserError):
            continue
    return [by_id[sid] for sid in selected if sid in by_id]


def upsert_batch_result(results, qa):
    sid = str(qa.get("study_id", ""))
    for idx, old in enumerate(results):
        if str(old.get("study_id", "")) == sid:
            results[idx] = qa
            return
    results.append(qa)


@contextmanager
def project_run_lock():
    lock_path = LOCAL_RESUME_DIR / "RUN.lock"
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    handle = lock_path.open("a+b")
    if handle.tell() == 0:
        handle.write(b"0")
        handle.flush()
    handle.seek(0)
    locked = False
    try:
        try:
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            locked = True
        except OSError as e:
            raise RuntimeError("Another patched extractor is running for this project on this computer. Stop it before starting another run.") from e
        yield
    finally:
        if locked:
            handle.seek(0)
            if os.name == "nt":
                import msvcrt
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else:
                import fcntl
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)
        handle.close()

print("Revision:", EXTRACTOR_REVISION)
print("Local checkpoints:", LOCAL_RESUME_DIR)
print("Local recoverable work cache:", LOCAL_WORK_DIR)


def restore_source_actions(path, selected):
    latest = {}
    for candidate in (path, _resume_local_path(path)):
        try:
            for row in pd.read_csv(candidate, dtype=str, keep_default_na=False).to_dict("records"):
                key = (str(row.get("study_id", "")), str(row.get("source_file_id", "")))
                if key[0] in selected and (key not in latest or _resume_timestamp(row) >= _resume_timestamp(latest[key])):
                    latest[key] = row
        except (OSError, ValueError, pd.errors.ParserError):
            continue
    return list(latest.values())


def upsert_source_action(rows, row):
    key = (row.get("study_id"), row.get("source_file_id"))
    for i, old in enumerate(rows):
        if (old.get("study_id"), old.get("source_file_id")) == key:
            rows[i] = row
            return
    rows.append(row)


_HASH_MEMORY = None

def cached_sha256(path):
    global _HASH_MEMORY
    if HASH_POLICY.upper() == "NONE":
        return ""
    if _HASH_MEMORY is None:
        _HASH_MEMORY = {(r["path"], r["size_bytes"], r["modified_time_ns"]): r["sha256"]
                        for r in load_hash_cache().to_dict("records")}
    stat = path.stat()
    key = (str(path), str(stat.st_size), str(stat.st_mtime_ns))
    if HASH_POLICY.upper() == "ONCE" and _HASH_MEMORY.get(key):
        return _HASH_MEMORY[key]
    digest = sha256_file(path)
    _HASH_MEMORY[key] = digest
    rows = [{"path":k[0],"size_bytes":k[1],"modified_time_ns":k[2],"sha256":v} for k,v in _HASH_MEMORY.items()]
    _write_resume_text(HASH_CACHE_PATH, pd.DataFrame(rows).to_csv(index=False))
    return digest


In [ ]:
# 17. Run maintenance OR the selected batch; preserve progress after every source.
report_stem = (f"batch_{BATCH_NO:03d}" if EXECUTION_MODE == "BATCH" else EXECUTION_MODE.lower()) + "__v4_3"
batch_progress_path = BATCH_REPORT_DIR / (report_stem + "__LIVE_PROGRESS.csv")
source_action_path = BATCH_REPORT_DIR / (report_stem + "__SOURCE_ACTIONS.csv")
batch_results = restore_batch_results(batch_progress_path, SELECTED_STUDIES)
source_actions = restore_source_actions(source_action_path, SELECTED_STUDIES)
if EXECUTION_MODE != "BATCH" and (FORCE_REPROCESS or REPARSE_SOURCE_IDS):
    raise ValueError("Forced reconversion is available only in BATCH mode")

with project_run_lock():
    for position, study_id in enumerate(SELECTED_STUDIES, 1):
        print(f"\n{EXECUTION_MODE}: {position}/{len(SELECTED_STUDIES)} | study {study_id}")
        study_out = RUN_EXTRACTED_DIR / safe_name(study_id)
        try:
            rows = source_rows_for_study(study_id)
            # Validate per-source checkpoints even during maintenance, not just report rows.
            if EXECUTION_MODE != "AUDIT_EXISTING" and study_checkpoint_is_current(study_id):
                cp = read_checkpoint(CHECKPOINT_DIR / f"{study_id}__STUDY_COMPLETE.json") or {}
                qa = {**cp.get("study_qa", {}), "resumed_from_checkpoint": True}
                print("  SKIP current study")
            else:
                source_qas = []
                for row in rows.to_dict("records"):
                    try:
                        source_qa = process_source(study_id, row, study_out)
                    except Exception as exc:
                        # An unreadable file does not prevent the remaining supplements from being processed.
                        source_qa = {"source_file_id": row.get("source_file_id"), "source_file_name": row.get("source_file_name"),
                                     "role": row.get("role"), "source_qa_status": "FAIL", "error": repr(exc),
                                     "traceback": traceback.format_exc(), "_action": "SOURCE_ERROR"}
                        print("  SOURCE ERROR:", row.get("source_file_name"), repr(exc))
                    source_qas.append(source_qa)
                    upsert_source_action(source_actions, {"study_id": study_id, "source_file_id": row.get("source_file_id"),
                                           "source_file_name": row.get("source_file_name"), "action": source_qa.get("_action"),
                                           "source_qa_status": source_qa.get("source_qa_status"),
                                           "issue": source_qa.get("issue") or source_qa.get("error") or "; ".join(map(str, source_qa.get("warnings") or [])),
                                           "needs_core_reparse": bool(source_qa.get("needs_core_reparse")),
                                           "completed_at_utc": datetime.now(timezone.utc).isoformat()})
                    if EXECUTION_MODE != "AUDIT_EXISTING":
                        _write_resume_text(source_action_path, pd.DataFrame(source_actions).to_csv(index=False))
                    gc.collect()
                qa = aggregate_study_qa(study_id, source_qas, rows)
                if EXECUTION_MODE != "AUDIT_EXISTING":
                    study_out.mkdir(parents=True, exist_ok=True)
                    write_json_atomic(study_out / "STUDY_QA.v4_3.json", qa)
                    write_study_handoff(study_out, study_id, qa, rows, source_qas)
                    write_checkpoint(CHECKPOINT_DIR / f"{study_id}__STUDY_COMPLETE.json", {
                        "study_id": study_id, "pipeline_version": PIPELINE_VERSION, "extractor_revision": EXTRACTOR_REVISION,
                        "patch_fingerprint": patch_fingerprint(), "source_signatures": study_current_signatures(study_id),
                        "study_qa": qa, "_resume_state": "COMPLETE", "completed_at_utc": datetime.now(timezone.utc).isoformat()})
            upsert_batch_result(batch_results, qa)
        except Exception as exc:
            traceback.print_exc()
            upsert_batch_result(batch_results, {"study_id": study_id, "study_source_qa_status": "FATAL_EXCEPTION",
                                                "error": repr(exc), "completed_at_utc": datetime.now(timezone.utc).isoformat()})
        if EXECUTION_MODE != "AUDIT_EXISTING":
            _write_resume_text(batch_progress_path, pd.DataFrame(batch_results).to_csv(index=False))
print("Run finished. A completed run does not mean every source passed QA.")


In [ ]:
# 18. Maintenance / batch QA summary
if batch_results:
    report_df = pd.DataFrame(batch_results)
    display(report_df)
    if "study_source_qa_status" in report_df:
        display(report_df["study_source_qa_status"].value_counts(dropna=False).to_frame("studies"))
print("Run report:", batch_progress_path)
print("Source action report:", source_action_path if EXECUTION_MODE != "AUDIT_EXISTING" else "in-memory only; AUDIT writes no report files")
print("Existing master database and original parsed outputs were not edited by maintenance.")


In [ ]:
# 19. Corpus dashboard: distinguish old QA from this patch's QA.
corpus_qa_rows = []
for study_dir in sorted(RUN_EXTRACTED_DIR.iterdir()):
    if not study_dir.is_dir():
        continue
    path = study_dir / "STUDY_QA.v4_3.json"
    if not path.exists():
        path = study_dir / "STUDY_QA.json"
    qa = read_json_if_exists(path)
    if isinstance(qa, dict):
        corpus_qa_rows.append({**qa, "_study_folder": study_dir.name,
                              "upgrade_current": qa.get("extractor_revision") == EXTRACTOR_REVISION})
corpus_qa_df = pd.DataFrame(corpus_qa_rows)
if not corpus_qa_df.empty:
    display(corpus_qa_df)
    if EXECUTION_MODE != "AUDIT_EXISTING":
        _write_resume_text(RUN_LOGS_DIR / "corpus_source_qa.v4_3.csv", corpus_qa_df.to_csv(index=False))
print("Dashboard includes legacy results; use upgrade_current to distinguish upgraded studies.")


In [ ]:
# 20. Only these sources require conversion/retry or human review.
if source_actions:
    actions_df = pd.DataFrame(source_actions)
    attention = actions_df[actions_df["action"].isin(["NEEDS_REPARSE", "SOURCE_ERROR", "MISSING_SOURCE", "SOURCE_CHANGED", "NEEDS_REVIEW"]) | actions_df["source_qa_status"].isin(["FAIL", "NEEDS_REVIEW"])]
    display(attention)
    print("Sources requiring attention:", len(attention))
else:
    print("No source actions in this run.")
